# SeamlessM4T v2 → Textless Pure S2ST (~673M)
## Voice Cloning · 5 Languages · Long-Form Audio · INTERSPEECH/IWSLT 2026

**Architectural transformation paper** — converting SeamlessM4T v2 from text-mediated S2ST
to a fully textless, speaker-preserving, long-form-capable S2ST system.

| Component | Original | After |
|---|---|---|
| Text Decoder | 867M, 24 layers | **0M — permanently removed** |
| lm_head + shared vocab | ~262M | **0M — removed** |
| Speech Encoder | 635M, 24L | **~441M, 16L** (BI+iterative prune) |
| T2U Model | 262M, 6+6L | **~175M, 4+4L** (LaCo RDSC merge) |
| CIF Connector | — | **~5M NEW** (trained from scratch) |
| Speaker Adapter | — | **~0.1M NEW** (ECAPA→vocoder 192→256) |
| **Total** | **1805M** | **~673M** |

**Papers:** S2UT (Lee ACL 2022) · SeamlessExpressive (arXiv:2312.05187) ·
LaCo (Yang EMNLP 2024) · CIF (Dong & Xu ICASSP 2020) · ECAPA-TDNN (Desplanques IS 2020) ·
DoRA (Liu ICML 2024) · ShortGPT (ACL 2025) · MMS (Pratap 2023)

**Phases:** P0 V1-Baseline → P1 Vocab5L → P2 EncPrune16L → P3 LaCoT2U →
P4 TextlessArch → P5 KD-Extract → P6a CIF-FeatureKD → P6b E2E-DoRA → P7 FullBenchmark


In [1]:
import os
# os.environ['CUDA_LAUNCH_BLOCKING'] = '1'
# ── OOM mitigation ────────────────────────────────────────────────────────────
# PyTorch's own error message recommended this.  Set before any CUDA allocations.
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"

# Lower the audio cap only if the no_grad fix alone is not enough.
# Fix 2 (no_grad on frozen conditioning) removes ~2-3 GB of activation graphs,
# which should be the entire OOM margin.  Start here; only drop further if needed.
# MAX_AUDIO_SEC_C = 10   # was 12 — safe starting point after the no_grad fix
#                         # raise back to 12 if you stay OOM-free for 300+ steps

In [2]:
# import os
# import signal

# # This kills the current Python process but keeps the Kaggle container alive.
# # The kernel will automatically restart, and the GPU memory WILL be cleared.
# os.kill(os.getpid(), signal.SIGKILL)

## ⚙️ Setup — run ALL at the start of EVERY Kaggle session

In [3]:
import os, sys, subprocess, pathlib, re, glob, json, gc, copy, time, math, shutil, random
import warnings; warnings.filterwarnings('ignore')

ON_KAGGLE = os.path.exists('/kaggle/working')
ON_COLAB  = not ON_KAGGLE
PLATFORM  = 'kaggle' if ON_KAGGLE else 'colab'

GDRIVE_MOUNT = '/content/drive/MyDrive/seamTL'   # ← NEW project folder
KAGGLE_WORK  = '/kaggle/working'

WORK_DIR  = KAGGLE_WORK if ON_KAGGLE else GDRIVE_MOUNT
CKPT_DIR  = f'{WORK_DIR}/checkpoints'
AUDIO_DIR = f'{WORK_DIR}/audio'
FIG_DIR   = f'{WORK_DIR}/figures'
MODEL_DIR = f'{WORK_DIR}/models'

GDRIVE_ROOT = 'gdrive:seamTL'   # rclone remote root (Kaggle only)


In [4]:
if ON_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    print(f'Drive mounted. Working folder: {GDRIVE_MOUNT}')
else:
    print('Kaggle: skipping Drive mount.')


Kaggle: skipping Drive mount.


In [5]:
for d in [WORK_DIR, CKPT_DIR, AUDIO_DIR, FIG_DIR, MODEL_DIR]:
    os.makedirs(d, exist_ok=True)
print(f'Platform : {PLATFORM}')
print(f'Work dir : {WORK_DIR}')
print(f'Checkpts : {CKPT_DIR}')


Platform : kaggle
Work dir : /kaggle/working
Checkpts : /kaggle/working/checkpoints


In [6]:
if ON_KAGGLE:
    subprocess.run('curl -s https://rclone.org/install.sh | sudo bash',
                   shell=True, capture_output=True)
    ver = subprocess.run('rclone version', shell=True, capture_output=True, text=True)
    print(ver.stdout.split('\n')[0])
else:
    print('Colab: rclone not needed — using mounted Drive directly.')
    if not os.path.exists('/content/drive/MyDrive'):
        print('WARNING: Drive does not appear to be mounted.')
    else:
        print('Drive mount: OK')


rclone v1.74.0


In [7]:
def _get_secret(key):
    if ON_KAGGLE:
        try:
            from kaggle_secrets import UserSecretsClient
            return UserSecretsClient().get_secret(key)
        except Exception as e:
            raise RuntimeError(f'Kaggle secret {key!r} not found: {e}')
    else:
        try:
            from google.colab import userdata
            return userdata.get(key)
        except Exception as e:
            raise RuntimeError(f'Colab secret {key!r} not found: {e}')

if ON_KAGGLE:
    RCLONE_CONF = _get_secret('RCLONE_CONF')
    raw = RCLONE_CONF.strip()
    raw = re.sub(r'\s*(\[[^\]]+\])\s*', r'\n\1\n', raw)
    raw = re.sub(r'\s+(type|scope|token|team_drive|client_id|client_secret|'
                 r'root_folder_id|service_account_file|drive_id)\s*=\s*',
                 r'\n\1 = ', raw)
    raw = raw.strip() + '\n'
    rclone_cfg = pathlib.Path.home() / '.config/rclone/rclone.conf'
    rclone_cfg.parent.mkdir(parents=True, exist_ok=True)
    rclone_cfg.write_text(raw)
    r = subprocess.run('rclone lsd gdrive:', shell=True, capture_output=True, text=True)
    print('Drive root:' if r.returncode == 0 else 'rclone FAILED:')
    print(r.stdout[:300] or r.stderr[:300])
else:
    print('Colab: skipping rclone config.')

try:
    HF_TOKEN = _get_secret('HF_TOKEN')
    from huggingface_hub import login
    login(HF_TOKEN)
    print('HuggingFace login: OK')
except Exception as e:
    print(f'HF login skipped: {e}')


Drive root:
           0 2026-04-17 11:03:10        -1 Colab Notebooks
           0 2025-11-10 11:33:43        -1 ScholarMate
           0 2026-04-05 12:59:09        -1 cse465
           0 2026-04-12 12:42:04        -1 cse465v5
           0 2026-04-23 09:47:12        -1 seamTL
           0 2026-04-23 20:52:48  
HuggingFace login: OK


In [8]:
# !rclone copy gdrive:seamTL gdrive:seamTLgoldBackup --drive-server-side-across-configs --transfers=32 --checkers=16

In [9]:
subprocess.run([
    'pip', 'install', '-q',
    'transformers>=4.41.0', 'datasets', 'torchaudio', 'speechbrain>=1.0.0',
    'peft>=0.10.0', 'librosa', 'jiwer', 'evaluate', 'sacrebleu', 'pyarrow',
    'sentencepiece', 'accelerate', 'matplotlib', 'seaborn',
    'soundfile', 'requests', 'pandas',
], check=True)
print('All packages installed.')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 69.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 7.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.8/100.8 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 83.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 45.0 MB/s eta 0:00:00
All packages installed.


In [10]:
from transformers.utils import logging
logging.set_verbosity_error()

In [11]:
import torch
import random
import numpy as np

autocast_dtype        = torch.float16
seed = 42

random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)

In [12]:
# ── pulled verbatim from seamless-cse465v5 Cell 14 ──
import torch
from datetime import datetime

_CUSTOM_STATE_FILE = '_custom_state.pt'
_PRUNING_MANIFEST  = 'pruning_manifest.pt'

def _rclone_push(local_path, remote_subpath):
    if not ON_KAGGLE: return
    r = subprocess.run(
        f'rclone copy "{local_path}" "{GDRIVE_ROOT}/{remote_subpath}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'[rclone] WARNING: push failed for {local_path}: {r.stderr[:200]}')

def _rclone_pull_model(stage_name):
    if not ON_KAGGLE: return
    local = f'{MODEL_DIR}/{stage_name}'
    os.makedirs(local, exist_ok=True)
    r = subprocess.run(
        f'rclone sync "{GDRIVE_ROOT}/models/{stage_name}/" "{local}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
        shell=True, capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError(f'[rclone] model pull failed for {stage_name}: {r.stderr[:300]}')
    print(f'[rclone] Pulled {stage_name} → {local}')

import threading
import queue

_upload_queue = queue.Queue()
_upload_thread = None

def _upload_worker():
    """Daemon worker that uploads checkpoints one at a time in background."""
    while True:
        item = _upload_queue.get()
        if item is None:  # poison pill
            break
        local_path, remote_subdir = item
        try:
            _rclone_push(local_path, remote_subdir)
        except Exception as e:
            print(f"[upload-bg] FAILED {os.path.basename(local_path)}: {e}")
        _upload_queue.task_done()

def _ensure_upload_thread():
    global _upload_thread
    if _upload_thread is None or not _upload_thread.is_alive():
        _upload_thread = threading.Thread(target=_upload_worker, daemon=True)
        _upload_thread.start()

def save_checkpoint(state, name, step=0, keep=3):
    fname = f'{name}_step{step:06d}.pt'
    path  = f'{CKPT_DIR}/{fname}'
    torch.save(state, path)
    mb = os.path.getsize(path) / 1e6
    print(f'[ckpt] Saved {fname} ({mb:.1f} MB)')
    
    if ON_KAGGLE:
        _ensure_upload_thread()
        _upload_queue.put((path, 'checkpoints'))
        print(f"[ckpt] Queued {fname} for background upload "
              f"({_upload_queue.qsize()} pending)")
    
    # Delete old local checkpoints (keep recent N)
    old = sorted(glob.glob(f'{CKPT_DIR}/{name}_step*.pt'))
    for f in old[:-keep]:
        if os.path.exists(f):
            os.remove(f)

def wait_for_uploads(timeout=300):
    """Optional: call at end of notebook to wait for pending uploads."""
    _upload_queue.join()
    print("[upload-bg] All uploads complete.")
    

def load_latest_checkpoint(name):
    files = sorted(glob.glob(f'{CKPT_DIR}/{name}_step*.pt'))
    if not files:
        print(f'[ckpt] No checkpoint for {name!r}')
        return None
    state = torch.load(files[-1], map_location='cpu', weights_only=False)
    print(f'[ckpt] Loaded {os.path.basename(files[-1])}')
    return state

def sync_checkpoints_from_drive():
    if ON_KAGGLE:
        print('[ckpt] Syncing from rclone remote...')
        r = subprocess.run(
            f'rclone sync "{GDRIVE_ROOT}/checkpoints/" "{CKPT_DIR}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
            shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'[ckpt] WARNING: {r.stderr[:300]}')
    else:
        print(f'[ckpt] Colab: reading directly from {CKPT_DIR}')
    files = sorted(os.listdir(CKPT_DIR)) if os.path.exists(CKPT_DIR) else []
    print(f'[ckpt] {len(files)} file(s) available')
    for f in files:
        mb = os.path.getsize(f'{CKPT_DIR}/{f}') / 1e6
        print(f'  {f:<55} {mb:>7.1f} MB')

print('Checkpoint helpers ready.')


Checkpoint helpers ready.


In [13]:
import torch.nn as nn, torch.nn.functional as F

_CUSTOM_ATTR_NAMES = ['_vocab_remap_to_old']

def _save_custom_state(mdl, path):
    state = {a: getattr(mdl, a) for a in _CUSTOM_ATTR_NAMES if hasattr(mdl, a)}
    if state:
        torch.save(state, os.path.join(path, _CUSTOM_STATE_FILE))
        print(f'  Saved custom state: {list(state.keys())}')

def _load_custom_state(mdl, path):
    fpath = os.path.join(path, _CUSTOM_STATE_FILE)
    if not os.path.exists(fpath): return
    state = torch.load(fpath, map_location='cpu', weights_only=False)
    for k, v in state.items(): setattr(mdl, k, v)
    print(f'  Restored custom state: {list(state.keys())}')

def _find_layers(component):
    for attr in ['layers', 'inner_layers', 'layer']:
        mod = getattr(component, attr, None)
        if isinstance(mod, nn.ModuleList) and len(mod) > 0:
            return mod
    return None

def _get_t2u_encoder_decoder(mdl):
    t2u   = getattr(mdl, 't2u_model', None)
    if t2u is None: return None, None
    inner = getattr(t2u, 'model', None)
    if inner is None: return None, None
    return getattr(inner, 'encoder', None), getattr(inner, 'decoder', None)

def sync_model_config(mdl):
    """Keep config in sync with actual ModuleList depths after pruning."""    
    cfg = mdl.config
    if hasattr(mdl, 'speech_encoder'):
        enc = mdl.speech_encoder
        parent = enc.encoder if hasattr(enc, 'encoder') else enc
        if hasattr(parent, 'layers'):
            actual = len(parent.layers)
            for k in ['speech_encoder_layers']:
                if hasattr(cfg, k) and getattr(cfg, k) != actual:
                    print(f'  [config] {k}: {getattr(cfg,k)} -> {actual}')
                    setattr(cfg, k, actual)
            sc = getattr(mdl.speech_encoder, 'config', None)
            if sc and hasattr(sc, 'num_hidden_layers') and sc.num_hidden_layers != actual:
                sc.num_hidden_layers = actual
    if hasattr(mdl, 'text_decoder') and mdl.text_decoder is not None:
        layers = _find_layers(mdl.text_decoder)
        if layers is not None:
            actual = len(layers)
            if hasattr(cfg, 'decoder_layers') and cfg.decoder_layers != actual:
                print(f'  [config] decoder_layers: {cfg.decoder_layers} -> {actual}')
                cfg.decoder_layers = actual
    t2u_enc, t2u_dec = _get_t2u_encoder_decoder(mdl)
    for sub, attr in [(t2u_enc,'t2u_encoder_layers'), (t2u_dec,'t2u_decoder_layers')]:
        if sub is None: continue
        layers = _find_layers(sub)
        if layers and hasattr(cfg, attr) and getattr(cfg, attr) != len(layers):
            print(f'  [config] {attr}: {getattr(cfg,attr)} -> {len(layers)}')
            setattr(cfg, attr, len(layers))
    t2u = getattr(mdl, 't2u_model', None)
    if t2u and hasattr(t2u, 'config'):
        tc = t2u.config
        for sub, attr in [(t2u_enc,'encoder_layers'), (t2u_dec,'decoder_layers')]:
            if sub is None: continue
            layers = _find_layers(sub)
            if layers and hasattr(tc, attr) and getattr(tc, attr) != len(layers):
                print(f'  [config] t2u.config.{attr}: {getattr(tc,attr)} -> {len(layers)}')
                setattr(tc, attr, len(layers))
    print('  [config] sync done.')

def _consolidate_to_single_gpu(mdl):
    """Move model to cuda:0 if split by device_map='auto'."""    
    if not torch.cuda.is_available(): return mdl
    if not (hasattr(mdl, 'hf_device_map') and len(set(mdl.hf_device_map.values())) > 1):
        return mdl
    print('  Multi-device → consolidating to cuda:0...')
    try:
        from accelerate.hooks import remove_hook_from_submodules
        remove_hook_from_submodules(mdl)
    except Exception: pass
    mdl = mdl.to('cuda:0')
    torch.cuda.empty_cache()
    print(f'  Model now on: {next(mdl.parameters()).device}')
    return mdl

def load_hf_weights_dict(model_dir):
    from pathlib import Path
    safe = Path(model_dir) / 'model.safetensors'
    if safe.is_file():
        try:
            from safetensors.torch import load_file
            return load_file(str(safe))
        except ImportError: pass
    pt = Path(model_dir) / 'pytorch_model.bin'
    if pt.is_file():
        blob = torch.load(str(pt), map_location='cpu', weights_only=False)
        return blob.get('model', blob) if isinstance(blob, dict) else blob
    return None

def _infer_t2u_layer_counts(model_dir):
    sd = load_hf_weights_dict(model_dir)
    if not sd: return None, None
    enc_idx, dec_idx = set(), set()
    for k in sd:
        if k.startswith('t2u_model.model.encoder.layers.'):
            r = k.split('.')[4]
            if r.isdigit(): enc_idx.add(int(r))
        elif k.startswith('t2u_model.model.decoder.layers.'):
            r = k.split('.')[4]
            if r.isdigit(): dec_idx.add(int(r))
    return (max(enc_idx)+1 if enc_idx else None), (max(dec_idx)+1 if dec_idx else None)

def save_model_to_drive(mdl, proc, stage_name, manifest_extra=None):
    """Save model to Drive using HF save_pretrained (battle-tested from v5)."""    
    target = f'{MODEL_DIR}/{stage_name}'
    os.makedirs(target, exist_ok=True)
    print(f'[model] Saving {stage_name} → {target} ...')
    sync_model_config(mdl)
    _save_custom_state(mdl, target)
    man = {'stage_name': stage_name}
    if manifest_extra: man.update(manifest_extra)
    torch.save(man, os.path.join(target, _PRUNING_MANIFEST))
    try:
        mdl.save_pretrained(target, safe_serialization=True)
    except Exception as e:
        print(f'  safe_serialization failed ({e}); trying .bin')
        mdl.save_pretrained(target)
    if proc is not None: proc.save_pretrained(target)
    total = sum(os.path.getsize(f'{target}/{f}') for f in os.listdir(target)) / 1e6
    print(f'[model] Local: {total:.0f} MB in {len(os.listdir(target))} files.')
    if ON_KAGGLE:
        r = subprocess.run(f'rclone sync "{target}/" "{GDRIVE_ROOT}/models/{stage_name}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
                           shell=True, capture_output=True, text=True)
        if r.returncode != 0:
            print(f'[model] WARNING rclone push failed: {r.stderr[:300]}')
        else:
            print(f'[model] Pushed to remote: {GDRIVE_ROOT}/models/{stage_name}/')
    else:
        print('[model] Colab: saved directly to Drive.')
        

def load_model_from_drive(stage_name, device_map=None):
    from transformers import SeamlessM4Tv2ForSpeechToSpeech, SeamlessM4TProcessor, AutoConfig
    local = f'{MODEL_DIR}/{stage_name}'
    if ON_KAGGLE and (not os.path.exists(local) or not os.listdir(local)):
        print(f'[model] Not in local cache — pulling from remote...')
        _rclone_pull_model(stage_name)
    if not os.path.exists(local) or not os.listdir(local):
        raise RuntimeError(f'[model] Not found or empty: {local}')
    wf = [f for f in os.listdir(local) if f.endswith('.safetensors') or f.endswith('.bin')]
    if not wf:
        raise RuntimeError(f'[model] No weight files in {local}')
    print(f'[model] Loading {stage_name} from {local} ...')
    cfg = AutoConfig.from_pretrained(local)
    enc_n, dec_n = _infer_t2u_layer_counts(local)
    if enc_n and getattr(cfg,'t2u_encoder_layers',None) != enc_n:
        print(f'  Repair T2U enc depth: {cfg.t2u_encoder_layers} -> {enc_n}')
        cfg.t2u_encoder_layers = enc_n
    if dec_n and getattr(cfg,'t2u_decoder_layers',None) != dec_n:
        print(f'  Repair T2U dec depth: {cfg.t2u_decoder_layers} -> {dec_n}')
        cfg.t2u_decoder_layers = dec_n
    mdl = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
        local, config=cfg, torch_dtype=torch.float16, device_map=device_map)
    _load_custom_state(mdl, local)
    proc = SeamlessM4TProcessor.from_pretrained(local)
    mdl.eval()
    print(f'[model] Loaded {stage_name}.')
    return mdl, proc

print('Model I/O helpers ready.')


Model I/O helpers ready.


In [14]:
import numpy as np, matplotlib.pyplot as plt, matplotlib, seaborn as sns
matplotlib.rcParams.update({'font.size': 11, 'figure.dpi': 120, 'savefig.bbox': 'tight'})
sns.set_style('whitegrid')

N_GPU = torch.cuda.device_count()
print(f'PyTorch {torch.__version__} | CUDA {torch.cuda.is_available()} | GPUs {N_GPU}')
for i in range(N_GPU):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU{i}: {torch.cuda.get_device_name(i)}  {p.total_memory/1e9:.1f} GB')

def count_params(module):
    return sum(p.numel() for p in module.parameters()) / 1e6

def count_params_detailed(model):
    bd = {n: count_params(c) for n, c in model.named_children()}
    bd['TOTAL'] = count_params(model)
    return bd

def print_model_breakdown(model, title='Model Breakdown'):
    bd = count_params_detailed(model)
    print(f'\n--- {title} ---')
    total = bd.pop('TOTAL')
    for name, p in sorted(bd.items(), key=lambda x: -x[1]):
        pct = p / total * 100 if total > 0 else 0
        print(f'  {name:<35} {p:>8.1f}M  ({pct:>5.1f}%)')
    print(f'  {"TOTAL":<35} {total:>8.1f}M')
    print('---')
    return {**bd, 'TOTAL': total}

def gpu_mem():
    if torch.cuda.is_available():
        for i in range(N_GPU):
            a = torch.cuda.memory_allocated(i)/1e9
            r = torch.cuda.memory_reserved(i)/1e9
            print(f'  GPU{i}: {a:.2f}GB alloc / {r:.2f}GB reserved')

def save_figure(fig, name):
    fig.savefig(f'{FIG_DIR}/{name}', dpi=150, bbox_inches='tight')
    if ON_KAGGLE: _rclone_push(f'{FIG_DIR}/{name}', 'figures')
    print(f'[fig] Saved {name}')

import torchaudio
from IPython.display import Audio as IPAudio, display

def play(audio, sr, label=''):
    if hasattr(audio, 'numpy'): audio = audio.squeeze().numpy()
    print(f'  {label}  ({len(audio)/sr:.1f}s | sr={sr})')
    display(IPAudio(audio, rate=int(sr)))

def save_audio(audio, sr, filename):
    path = f'{AUDIO_DIR}/{filename}'
    if not isinstance(audio, torch.Tensor): audio = torch.tensor(audio)
    torchaudio.save(path, audio.squeeze().unsqueeze(0).float().cpu(), sr)
    print(f'[audio] Saved {filename}')

print('Core utilities ready.')


PyTorch 2.10.0+cu128 | CUDA True | GPUs 2
  GPU0: Tesla T4  15.6 GB
  GPU1: Tesla T4  15.6 GB
Core utilities ready.


In [15]:
def _load_summaries_from_drive():
    ckpt = load_latest_checkpoint('all_summaries')
    if ckpt and 'summaries' in ckpt:
        return {s['label']: s for s in ckpt['summaries']}
    return {}

ALL_SUMMARIES: dict = _load_summaries_from_drive()
print(f'Loaded {len(ALL_SUMMARIES)} existing summaries: {list(ALL_SUMMARIES.keys())}')

def store_summary(s):
    label = s['label']
    ALL_SUMMARIES[label] = s.copy()
    save_checkpoint({'summaries': list(ALL_SUMMARIES.values())}, 'all_summaries', 0)
    print(f'[summary] Stored {label} ({len(ALL_SUMMARIES)} total)')

def get_summaries():
    return sorted(ALL_SUMMARIES.values(), key=lambda s: s['label'])

def plot_phase_comparison(summaries=None, save_name='phase_comparison.png'):
    data = summaries or get_summaries()
    if not data: 
        print('No summaries yet.'); 
        return
    
    # Sort by label to ensure consistent ordering
    data = sorted(data, key=lambda s: s['label'])
    labels = [s['label'] for s in data]
    
    print(f'Plotting {len(data)} phases: {labels}')
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 10))
    fig.suptitle('Textless S2ST Compression Pipeline: Phase Comparison',
                 fontsize=15, fontweight='bold')
    metrics = [('avg_bleu', 'ASR-BLEU (higher=better)', '#2196F3'),
               ('avg_chrf', 'ASR-ChrF (higher=better)', '#4CAF50'),
               ('avg_rtf',  'RTF (lower=faster)',        '#FF9800'),
               ('params_M', 'Parameters (M)',            '#9C27B0')]
    
    for ax, (key, title, color) in zip(axes.flat, metrics):
        vals = [s.get(key, 0) for s in data]
        x_pos = range(len(labels))
        bars = ax.bar(x_pos, vals, color=color, alpha=0.85, edgecolor='white', width=0.7)
        ax.set_title(title, fontweight='bold', fontsize=11)
        ax.set_xticks(x_pos)
        ax.set_xticklabels(labels, rotation=40, ha='right', fontsize=8)
        ax.grid(axis='y', alpha=0.3, linestyle='--')
        
        # Add value labels on bars
        for bar, v in zip(bars, vals):
            height = bar.get_height()
            if height > 0:
                ax.text(bar.get_x() + bar.get_width()/2, height, 
                       f'{v:.1f}', ha='center', va='bottom', fontsize=7, fontweight='bold')
    
    plt.tight_layout()
    save_figure(fig, save_name)
    plt.show()
def plot_size_vs_quality(summaries=None, save_name='size_vs_quality.png'):
    data = summaries or get_summaries()
    if not data: return
    fig, ax = plt.subplots(figsize=(10, 7))
    params = [s['params_M'] for s in data]
    chrf   = [s['avg_chrf'] for s in data]
    bleu   = [s['avg_bleu'] for s in data]
    ax.scatter(params, bleu, s=120, c='#2196F3', zorder=5, label='ASR-BLEU')
    ax.scatter(params, chrf, s=120, c='#4CAF50', marker='s', zorder=5, label='ASR-ChrF')
    for i, lbl in enumerate([s['label'] for s in data]):
        ax.annotate(lbl, (params[i], bleu[i]), fontsize=7, xytext=(5,5),
                    textcoords='offset points')
    ax.set_xlabel('Parameters (M)'); ax.set_ylabel('Score')
    ax.set_title('Model Size vs Translation Quality', fontweight='bold')
    ax.legend()
    plt.tight_layout()
    save_figure(fig, save_name)
    plt.show()

print('Plotting helpers ready.')


[ckpt] No checkpoint for 'all_summaries'
Loaded 0 existing summaries: []
Plotting helpers ready.


In [16]:
"""
QUICK INTEGRATION SNIPPET
Copy-paste this entire cell into seamless-final.ipynb after the existing summary functions
This is a condensed version for immediate use
"""

# ============================================================================
# ENHANCED TRACKING - Insert after ALL_SUMMARIES definition
# ============================================================================

def _load_detailed_summaries_from_drive():
    ckpt = load_latest_checkpoint('all_detailed_summaries')
    if ckpt and 'detailed_summaries' in ckpt:
        return {s['label']: s for s in ckpt['detailed_summaries']}
    return {}

ALL_DETAILED_SUMMARIES = _load_detailed_summaries_from_drive()
print(f'Loaded {len(ALL_DETAILED_SUMMARIES)} detailed summaries')

def store_detailed_summary(s):
    label = s['label']
    ALL_DETAILED_SUMMARIES[label] = s.copy()
    save_checkpoint({'detailed_summaries': list(ALL_DETAILED_SUMMARIES.values())}, 
                    'all_detailed_summaries', 0)
    print(f'[detailed] Stored {label}')

def compute_detailed_summary(results, label, params_M):
    from collections import defaultdict
    by_pair = defaultdict(list)
    for r in results:
        if not math.isnan(r.get('rtf', float('nan'))):
            by_pair[f"{r['src_lang']}→{r['tgt_lang']}"].append(r)
    
    pair_stats = {}
    for pair_key, pair_results in by_pair.items():
        pair_stats[pair_key] = {
            'n_samples': len(pair_results),
            'avg_bleu': float(np.mean([r['bleu'] for r in pair_results])),
            'avg_chrf': float(np.mean([r['chrf'] for r in pair_results])),
            'avg_rtf': float(np.mean([r['rtf'] for r in pair_results])),
            'std_chrf': float(np.std([r['chrf'] for r in pair_results])),
        }
    
    valid = [r for r in results if not math.isnan(r.get('rtf', float('nan')))]
    by_src = defaultdict(list)
    by_tgt = defaultdict(list)
    for r in valid:
        by_src[r['src_lang']].append(r)
        by_tgt[r['tgt_lang']].append(r)
    
    return {
        'label': label, 'params_M': params_M, 'n_total': len(valid),
        'avg_bleu': float(np.mean([r['bleu'] for r in valid])),
        'avg_chrf': float(np.mean([r['chrf'] for r in valid])),
        'avg_rtf': float(np.mean([r['rtf'] for r in valid])),
        'std_chrf': float(np.std([r['chrf'] for r in valid])),
        'pair_stats': pair_stats,
        'by_src_lang': {lang: {
            'n_samples': len(rs),
            'avg_chrf': float(np.mean([r['chrf'] for r in rs])),
            'avg_bleu': float(np.mean([r['bleu'] for r in rs])),
        } for lang, rs in by_src.items()},
        'by_tgt_lang': {lang: {
            'n_samples': len(rs),
            'avg_chrf': float(np.mean([r['chrf'] for r in rs])),
            'avg_bleu': float(np.mean([r['bleu'] for r in rs])),
        } for lang, rs in by_tgt.items()},
    }

def plot_detailed_phase_comparison(save_name='detailed_comparison.png'):
    summaries = sorted(ALL_DETAILED_SUMMARIES.values(), key=lambda s: s['label'])
    if not summaries: 
        print('No detailed summaries yet.')
        return
    
    print(f'Plotting detailed comparison for {len(summaries)} phases: {[s["label"] for s in summaries]}')
    
    fig = plt.figure(figsize=(20, 14))
    fig.suptitle('Detailed Phase Comparison: Per-Language Breakdown', fontsize=14, fontweight='bold')
    
    labels = [s['label'] for s in summaries]
    
    # Panel 1: Overall ChrF/BLEU
    ax1 = plt.subplot(3, 3, 1)
    chrfs = [s['avg_chrf'] for s in summaries]
    bleus = [s['avg_bleu'] for s in summaries]
    x = np.arange(len(labels))
    ax1.bar(x - 0.2, chrfs, 0.4, label='ChrF', color='#4CAF50', alpha=0.85)
    ax1.bar(x + 0.2, bleus, 0.4, label='BLEU', color='#2196F3', alpha=0.85)
    ax1.set_xticks(x)
    ax1.set_xticklabels(labels, rotation=30, ha='right', fontsize=7)
    ax1.set_title('Overall Quality', fontweight='bold')
    ax1.legend()
    ax1.grid(alpha=0.3)
    
    # Panel 2: Per-pair ChrF comparison across ALL phases (vertical bars, grouped by language pair)
    ax2 = plt.subplot(3, 3, 2)
    
    # Collect all unique language pairs across all phases
    all_pairs = set()
    for s in summaries:
        if 'pair_stats' in s:
            all_pairs.update(s['pair_stats'].keys())
    all_pairs = sorted(all_pairs)
    
    if all_pairs:
        n_pairs = len(all_pairs)
        n_phases = len(summaries)
        bar_width = 0.8 / n_phases
        x_pos = np.arange(n_pairs)
        
        for phase_idx, s in enumerate(summaries):
            pair_stats = s.get('pair_stats', {})
            chrf_vals = [pair_stats.get(pair, {}).get('avg_chrf', 0) for pair in all_pairs]
            offset = (phase_idx - n_phases/2 + 0.5) * bar_width
            ax2.bar(x_pos + offset, chrf_vals, bar_width, 
                   label=s['label'], alpha=0.85)
        
        ax2.set_xticks(x_pos)
        ax2.set_xticklabels(all_pairs, rotation=45, ha='right', fontsize=6)
        ax2.set_ylabel('ASR-ChrF')
        ax2.set_title('ChrF by Language Pair (All Phases)', fontweight='bold', fontsize=9)
        ax2.legend(fontsize=6, ncol=2)
        ax2.grid(alpha=0.3, axis='y')
    
    # Panel 3: BLEU by pair for all phases
    ax3 = plt.subplot(3, 3, 3)
    if all_pairs:
        for phase_idx, s in enumerate(summaries):
            pair_stats = s.get('pair_stats', {})
            bleu_vals = [pair_stats.get(pair, {}).get('avg_bleu', 0) for pair in all_pairs]
            offset = (phase_idx - n_phases/2 + 0.5) * bar_width
            ax3.bar(x_pos + offset, bleu_vals, bar_width, 
                   label=s['label'], alpha=0.85)
        
        ax3.set_xticks(x_pos)
        ax3.set_xticklabels(all_pairs, rotation=45, ha='right', fontsize=6)
        ax3.set_ylabel('ASR-BLEU')
        ax3.set_title('BLEU by Language Pair (All Phases)', fontweight='bold', fontsize=9)
        ax3.legend(fontsize=6, ncol=2)
        ax3.grid(alpha=0.3, axis='y')
    
    # Panel 4: Source language trends
    ax4 = plt.subplot(3, 3, 4)
    if summaries and 'by_src_lang' in summaries[0]:
        # Get all source languages
        all_src_langs = set()
        for s in summaries:
            if 'by_src_lang' in s:
                all_src_langs.update(s['by_src_lang'].keys())
        all_src_langs = sorted(all_src_langs)
        
        for src in all_src_langs:
            src_chrfs = []
            for s in summaries:
                if 'by_src_lang' in s and src in s['by_src_lang']:
                    src_chrfs.append(s['by_src_lang'][src]['avg_chrf'])
                else:
                    src_chrfs.append(None)
            valid_x = [i for i, v in enumerate(src_chrfs) if v is not None]
            valid_y = [v for v in src_chrfs if v is not None]
            if valid_y:
                ax4.plot(valid_x, valid_y, 'o-', label=src.upper(), lw=2, ms=5)
        ax4.set_xticks(range(len(labels)))
        ax4.set_xticklabels(labels, rotation=30, ha='right', fontsize=6)
        ax4.set_ylabel('ASR-ChrF')
        ax4.set_title('Source Language Trends', fontweight='bold', fontsize=9)
        ax4.legend(fontsize=6, ncol=2)
        ax4.grid(alpha=0.3)
    
    # Panel 5: Target language trends
    ax5 = plt.subplot(3, 3, 5)
    if summaries and 'by_tgt_lang' in summaries[0]:
        all_tgt_langs = set()
        for s in summaries:
            if 'by_tgt_lang' in s:
                all_tgt_langs.update(s['by_tgt_lang'].keys())
        all_tgt_langs = sorted(all_tgt_langs)
        
        for tgt in all_tgt_langs:
            tgt_chrfs = []
            for s in summaries:
                if 'by_tgt_lang' in s and tgt in s['by_tgt_lang']:
                    tgt_chrfs.append(s['by_tgt_lang'][tgt]['avg_chrf'])
                else:
                    tgt_chrfs.append(None)
            valid_x = [i for i, v in enumerate(tgt_chrfs) if v is not None]
            valid_y = [v for v in tgt_chrfs if v is not None]
            if valid_y:
                ax5.plot(valid_x, valid_y, 's-', label=tgt.upper(), lw=2, ms=5)
        ax5.set_xticks(range(len(labels)))
        ax5.set_xticklabels(labels, rotation=30, ha='right', fontsize=6)
        ax5.set_ylabel('ASR-ChrF')
        ax5.set_title('Target Language Trends', fontweight='bold', fontsize=9)
        ax5.legend(fontsize=6, ncol=2)
        ax5.grid(alpha=0.3)
    
    # Panel 6: Params vs Quality
    ax6 = plt.subplot(3, 3, 6)
    params = [s['params_M'] for s in summaries]
    ax6.scatter(params, chrfs, s=100, c='#4CAF50', marker='o', label='ChrF', zorder=5)
    ax6.scatter(params, bleus, s=100, c='#2196F3', marker='s', label='BLEU', zorder=5)
    for i, lbl in enumerate(labels):
        ax6.annotate(lbl, (params[i], chrfs[i]), fontsize=6, xytext=(3,3),
                    textcoords='offset points')
    ax6.set_xlabel('Parameters (M)')
    ax6.set_ylabel('Score')
    ax6.set_title('Size vs Quality', fontweight='bold', fontsize=9)
    ax6.legend(fontsize=7)
    ax6.grid(alpha=0.3)
    
    # Panel 7: Speaker sim by pair (if available)
    ax7 = plt.subplot(3, 3, 7)
    ax7.text(0.5, 0.5, 'Reserved for\nSpeaker Similarity', 
            ha='center', va='center', transform=ax7.transAxes, fontsize=10)
    ax7.axis('off')
    
    # Panel 8: RTF comparison
    ax8 = plt.subplot(3, 3, 8)
    rtfs = [s['avg_rtf'] for s in summaries]
    bars = ax8.bar(range(len(labels)), rtfs, color='#FF9800', alpha=0.85, edgecolor='white')
    ax8.set_xticks(range(len(labels)))
    ax8.set_xticklabels(labels, rotation=30, ha='right', fontsize=7)
    ax8.set_ylabel('RTF (lower=faster)')
    ax8.set_title('Inference Speed', fontweight='bold', fontsize=9)
    ax8.grid(alpha=0.3, axis='y')
    for bar, v in zip(bars, rtfs):
        if v > 0:
            ax8.text(bar.get_x()+bar.get_width()/2, bar.get_height(), 
                    f'{v:.3f}', ha='center', va='bottom', fontsize=6)
    
    # Panel 9: Summary table
    ax9 = plt.subplot(3, 3, 9)
    ax9.axis('off')
    table_data = [['Phase', 'Params(M)', 'ChrF', 'BLEU', 'RTF']]
    for s in summaries:
        table_data.append([
            s['label'][:12],
            f"{s['params_M']:.0f}",
            f"{s['avg_chrf']:.1f}",
            f"{s['avg_bleu']:.1f}",
            f"{s['avg_rtf']:.3f}"
        ])
    tbl = ax9.table(cellText=table_data[1:], colLabels=table_data[0],
                   cellLoc='center', loc='center')
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(7)
    tbl.scale(1.0, 1.5)
    ax9.set_title('Summary Table', fontweight='bold', fontsize=9, pad=10)
    
    plt.tight_layout()
    save_figure(fig, save_name)
    plt.show()
    print(f'✓ Detailed comparison plotted for {len(summaries)} phases')
def print_detailed_summary_table(phase_label=None):
    summaries = sorted(ALL_DETAILED_SUMMARIES.values(), key=lambda s: s['label'])
    if not summaries: return
    summary = next((s for s in summaries if s['label'] == phase_label), summaries[-1]) if phase_label else summaries[-1]
    
    print(f'\n{"="*80}\n  {summary["label"]} - {summary["params_M"]:.1f}M params\n{"="*80}')
    print(f'Overall: ChrF={summary["avg_chrf"]:.2f}±{summary.get("std_chrf",0):.2f}  '
          f'BLEU={summary["avg_bleu"]:.2f}  RTF={summary["avg_rtf"]:.4f}')
    
    pair_stats = summary.get('pair_stats', {})
    if pair_stats:
        print(f'\nPer-Pair ({len(pair_stats)} pairs):')
        print(f'  {"Pair":<15} {"N":>4} {"ChrF":>8} {"BLEU":>8} {"RTF":>8}')
        for pair in sorted(pair_stats.keys()):
            s = pair_stats[pair]
            print(f'  {pair:<15} {s["n_samples"]:>4} {s["avg_chrf"]:>8.2f} '
                  f'{s["avg_bleu"]:>8.2f} {s["avg_rtf"]:>8.4f}')
    
    by_src = summary.get('by_src_lang', {})
    if by_src:
        print(f'\nBy Source Language:')
        for lang in sorted(by_src.keys()):
            s = by_src[lang]
            print(f'  {lang.upper():>6}: ChrF={s["avg_chrf"]:>6.2f}  BLEU={s["avg_bleu"]:>6.2f}  (n={s["n_samples"]})')
    
    by_tgt = summary.get('by_tgt_lang', {})
    if by_tgt:
        print(f'\nBy Target Language:')
        for lang in sorted(by_tgt.keys()):
            s = by_tgt[lang]
            print(f'  {lang.upper():>6}: ChrF={s["avg_chrf"]:>6.2f}  BLEU={s["avg_bleu"]:>6.2f}  (n={s["n_samples"]})')
    print('='*80)

print('✓ Enhanced tracking loaded: store_detailed_summary(), compute_detailed_summary(), plot_detailed_phase_comparison(), print_detailed_summary_table()')


[ckpt] No checkpoint for 'all_detailed_summaries'
Loaded 0 detailed summaries
✓ Enhanced tracking loaded: store_detailed_summary(), compute_detailed_summary(), plot_detailed_phase_comparison(), print_detailed_summary_table()


## 📊 Enhanced Per-Language Tracking Enabled

**New functions available:**
- `compute_detailed_summary(results, label, params_M)` - Extract per-language metrics
- `store_detailed_summary(summary)` - Save to checkpoint
- `plot_detailed_phase_comparison()` - 9-panel visualization
- `print_detailed_summary_table(phase_label)` - Text output

**To use in benchmark cells:**
```python
# After running benchmark
p0_results, p0_summary = run_benchmark_asr(model, samples, 'P0_Label', save_n=4)
p0_detailed = compute_detailed_summary(p0_results, 'P0_Label', p0_summary['params_M'])

# Save both
save_checkpoint({
    'results': p0_results,
    'summary': p0_summary,
    'detailed_summary': p0_detailed  # NEW
}, 'phase0_benchmark', 0)

store_summary(p0_summary)
store_detailed_summary(p0_detailed)  # NEW
print_detailed_summary_table('P0_Label')  # NEW
plot_detailed_phase_comparison()  # NEW
```

**All per-language data now preserved in checkpoints!**

In [17]:
# ── MMS-ASR for Bengali, Hindi, Arabic ──────────────────────────────────────
import gc as _stdlib_gc

_MMS_MODEL_ID = 'facebook/mms-1b-all'
_mms_asr_models = {}  # Cache models per language
_mms_asr_processors = {}

def _ensure_mms_loaded(lang_code):
    """Load MMS model for specific language (ben, hin, ara)"""
    global _mms_asr_models, _mms_asr_processors
    if lang_code in _mms_asr_models: return
    
    from transformers import Wav2Vec2ForCTC, AutoProcessor
    print(f'[MMS-ASR] Loading {_MMS_MODEL_ID} lang={lang_code}...')
    _mms_asr_processors[lang_code] = AutoProcessor.from_pretrained(
        _MMS_MODEL_ID, target_lang=lang_code)
    _mms_asr_models[lang_code] = Wav2Vec2ForCTC.from_pretrained(
        _MMS_MODEL_ID, target_lang=lang_code,
        ignore_mismatched_sizes=True, torch_dtype=torch.float16)
    _mms_asr_models[lang_code].load_adapter(lang_code)
    _mms_asr_models[lang_code] = _mms_asr_models[lang_code].eval()
    try: 
        _mms_asr_models[lang_code] = _mms_asr_models[lang_code].to('cuda:0')
    except RuntimeError: 
        pass
    print(f'[MMS-ASR] {lang_code} ready.')

def asr_transcribe_mms(audio_np, lang_code, sr=16000):
    _ensure_mms_loaded(lang_code)
    if audio_np is None or len(audio_np) < 400:
        return ''

    if sr != 16000:
        audio_np = torchaudio.functional.resample(
            torch.tensor(audio_np), sr, 16000).numpy()

    model = _mms_asr_models[lang_code]
    processor = _mms_asr_processors[lang_code]

    device = next(model.parameters()).device
    dtype = next(model.parameters()).dtype

    inputs = processor(audio_np, sampling_rate=16000, return_tensors='pt')
    input_values = inputs.input_values.to(device).to(dtype)

    with torch.no_grad():
        logits = model(input_values=input_values).logits

    pred_ids = torch.argmax(logits, dim=-1)
    return processor.batch_decode(pred_ids)[0].strip()

# ── Whisper-medium for English and Chinese ──────────────────────────────────
_whisper_model = None
_whisper_processor = None

def _ensure_whisper_loaded():
    global _whisper_model, _whisper_processor
    if _whisper_model is not None: return
    from transformers import WhisperForConditionalGeneration, WhisperProcessor
    print('[Whisper] Loading openai/whisper-medium...')
    _whisper_processor = WhisperProcessor.from_pretrained('openai/whisper-medium')
    _whisper_model = WhisperForConditionalGeneration.from_pretrained(
        'openai/whisper-medium', torch_dtype=torch.float16)
    _whisper_model = _whisper_model.eval()
    try:
        device = 'cuda:1' if N_GPU > 1 else 'cuda:0'
        _whisper_model = _whisper_model.to(device)
    except RuntimeError:
        pass
    print('[Whisper] Ready.')

def asr_transcribe_whisper(audio_np, lang='en', sr=16000):
    """
    Transcribe audio using Whisper-medium for English or Chinese.
    lang: 'en' for English, 'zh' for Chinese
    """
    _ensure_whisper_loaded()
    if audio_np is None or len(audio_np) < 400: return ''
    
    # Resample if needed
    if sr != 16000:
        audio_np = torchaudio.functional.resample(
            torch.tensor(audio_np), sr, 16000).numpy()
    
    device = next(_whisper_model.parameters()).device
    dtype = next(_whisper_model.parameters()).dtype
    
    # Whisper language codes
    whisper_lang = 'en'
    
    try:
        # Process audio - ensure correct dtype
        inputs = _whisper_processor(
            audio_np, 
            sampling_rate=16000, 
            return_tensors='pt',
            return_attention_mask=True)
        
        # Move to device and convert to model dtype
        input_features = inputs['input_features'].to(device).to(dtype)
        
        # Use modern task/language parameters instead of forced_decoder_ids
        with torch.no_grad():
            predicted_ids = _whisper_model.generate(
                input_features,
                language=whisper_lang,
                task='transcribe',
                max_new_tokens=256,
                num_beams=1,
                do_sample=False)
        
        transcription = _whisper_processor.batch_decode(
            predicted_ids, skip_special_tokens=True)[0]
        return transcription.strip()
    except Exception as e:
        print(f'[Whisper] Error: {e}')
        import traceback
        traceback.print_exc()
        return ''

# ── M4T lang → ASR backend mapping ──────────────────────────────────────────
M4T_FLEURS_MAP = {
    'eng': 'en_us', 'ben': 'bn_in', 'cmn': 'cmn_hans_cn',
    'arb': 'ar_eg', 'hin': 'hi_in',
}

# MMS language codes - UPDATED to include Chinese
MMS_LANG_MAP = {
    'ben': 'ben',  # Bengali
    'hin': 'hin',  # Hindi
    'arb': 'ara',  # Arabic (MMS uses 'ara' for Arabic)
    'cmn': 'cmn',  # Chinese Mandarin (ADDED)
}

# UPDATED: Whisper only for English, MMS for all others
LANG_ASR_CONFIG = {
    'ben': ('mms', 'ben'),       # MMS for Bengali
    'hin': ('mms', 'hin'),       # MMS for Hindi
    'arb': ('mms', 'ara'),       # MMS for Arabic
    'cmn': ('mms', 'cmn-script_simplified'),       # MMS for Chinese (CHANGED from Whisper)
    'eng': ('whisper', 'en'),    # Whisper for English only
}

def asr_transcribe(audio_np, tgt_lang_m4t, sr=16000):
    """Route to correct ASR backend: Whisper for EN only, MMS for all others"""    
    if audio_np is None or len(audio_np) < 800: return ''
    backend, lang_code = LANG_ASR_CONFIG.get(tgt_lang_m4t)  # Default to MMS
    try:
        if backend == 'mms':
            return asr_transcribe_mms(audio_np, lang_code, sr)
        else:  # whisper (only for English now)
            return asr_transcribe_whisper(audio_np, lang_code, sr)
    except Exception as e:
        print(f'[ASR] Error ({tgt_lang_m4t}): {e}')
        return ''


print('ASR stack ready:')
print('  - Whisper-medium: English only')
print('  - MMS-1b-all: Bengali, Hindi, Arabic, Chinese')

ASR stack ready:
  - Whisper-medium: English only
  - MMS-1b-all: Bengali, Hindi, Arabic, Chinese


In [18]:
from sacrebleu.metrics import BLEU, CHRF
_bleu = BLEU(effective_order=True)
_chrf = CHRF()

def find_layers_attr(component):
    for attr in ['layers', 'layer', 'inner_layers', 'encoder_layers', 'decoder_layers']:
        if hasattr(component, attr): return attr
    return None

def compute_bleu(hyp, ref):
    if not hyp.strip() or not ref.strip(): return 0.0
    return _bleu.sentence_score(hyp.strip(), [ref.strip()]).score

def compute_chrf(hyp, ref):
    if not hyp.strip() or not ref.strip(): return 0.0
    return _chrf.sentence_score(hyp.strip(), [ref.strip()]).score

def _remap_ids_for_decode(mdl, ids):
    if hasattr(mdl, '_vocab_remap_to_old'):
        remap = mdl._vocab_remap_to_old
        ids = ids.clone()
        mask = (ids >= 0) & (ids < len(remap))
        ids[mask] = remap[ids[mask]]
    return ids

def _model_input_device(mdl):
    if hasattr(mdl, 'speech_encoder'):
        return next(mdl.speech_encoder.parameters()).device
    return next(mdl.parameters()).device

def run_s2st(mdl, wav, tgt_lang='ben'):
    """Full S2ST for models with text decoder (Phases 0-3)."""
    inputs = processor(audio=wav, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(_model_input_device(mdl)) for k, v in inputs.items()}
    with torch.no_grad():
        try:
            with torch.cuda.amp.autocast(dtype=autocast_dtype):   # ← FIX
                out = mdl.generate(**inputs, tgt_lang=tgt_lang,
                                   return_intermediate_token_ids=True)
            text_ids = _remap_ids_for_decode(mdl, out.sequences.cpu())
            text = processor.batch_decode(text_ids, skip_special_tokens=True)[0]
            wav_out = out.waveform.cpu().numpy().squeeze() if out.waveform is not None else np.zeros(16000)
            return text, wav_out
        except RuntimeError:
            return run_s2t_only(mdl, wav, tgt_lang), np.zeros(16000)


def run_s2t_only(mdl, wav, tgt_lang='ben'):
    """Text-only generation (for benchmarking text-decoder models)."""
    inputs = processor(audio=wav, sampling_rate=16000, return_tensors='pt')
    inputs = {k: v.to(_model_input_device(mdl)) for k, v in inputs.items()}
    orig_voc = mdl.vocoder
    inp_dev  = next(iter(inputs.values())).device
    class _Noop(nn.Module):
        def forward(self, *a, **kw): return torch.zeros(1,1,device=inp_dev), [1]
    mdl.vocoder = _Noop()
    try:
        with torch.no_grad():
            with torch.cuda.amp.autocast(dtype=autocast_dtype):   # ← FIX
                out = mdl.generate(**inputs, tgt_lang=tgt_lang,
                                   return_intermediate_token_ids=True)
    finally:
        mdl.vocoder = orig_voc
    text_ids = _remap_ids_for_decode(mdl, out.sequences.cpu())
    return processor.batch_decode(text_ids, skip_special_tokens=True)[0]

def quick_eval_chrf(mdl, samples, max_samples=16, group_size=25):
    """
    Optimized: Only load audio for samples we actually use.
    """
    text_scores = []
    asr_scores = []
    num_langs = len(samples) // group_size
    per_lang = max(1, max_samples // num_langs)
    
    for i in range(num_langs):
        start = i * group_size
        
        # ✅ OPTIMIZED: Only load the samples we need
        for j in range(per_lang):
            
            idx = start + j
            if idx >= len(samples):
                break
            
            s = samples[idx]  # Load only this one sample
            tgt = s.get('tgt_lang', 'ben')
            text_pred, wav_out = run_s2st(mdl, s['wav'], tgt_lang=tgt)
            asr_pred = asr_transcribe(wav_out, tgt)

            text_scores.append(compute_chrf(text_pred, s['ref']))
            asr_scores.append(compute_chrf(asr_pred, s['ref']))
    
    return float(np.mean(text_scores)), float(np.mean(asr_scores))

print('Benchmark functions ready.')

Benchmark functions ready.


In [19]:
import jieba

def zh_tokenize(text):
    return " ".join(jieba.lcut(text.replace(" ", "")))


def run_benchmark_asr(mdl, samples, label='model', save_n=4):
    """ASR-based benchmark: translate audio → ASR transcribe → compute ASR-ChrF/BLEU."""
    print(f'\n{"="*60}\n  BENCHMARK (ASR): {label}  Samples:{len(samples)}\n{"="*60}')
    gpu_mem()
    results = []
    
    # Group samples by language pair for organized output
    from collections import defaultdict
    by_pair = defaultdict(list)
    for s in samples:
        by_pair[f"{s['src_lang']}→{s['tgt_lang']}"].append(s)
    
    for pair_key, pair_samples in by_pair.items():
        print(f'\n  === {pair_key} ({len(pair_samples)} samples) ===')
        for i, s in enumerate(pair_samples):
            try:
                dur = len(s['wav']) / 16000
                t0  = time.time()
                # Run S2ST translation
                _, wav_out = run_s2st(mdl, s['wav'], tgt_lang=s['tgt_lang'])
                rtf  = (time.time() - t0) / dur
                
                # ASR transcribe output audio
                pred = asr_transcribe(wav_out, s['tgt_lang'])
                
                ref = s['ref']
                hyp = pred
                if s['tgt_lang'] == 'cmn':
                    print("bench cmn")
                    ref_clean = ref.replace(" ", "")
                    hyp_clean = hyp.replace(" ", "")
                
                    # BLEU (tokenized)
                    ref_bleu = zh_tokenize(ref_clean)
                    hyp_bleu = zh_tokenize(hyp_clean)
                
                    # chrF (raw)
                    ref_chrf = ref_clean
                    hyp_chrf = hyp_clean

                    bleu = compute_bleu(hyp_bleu, ref_bleu)
                    chrf = compute_chrf(hyp_chrf, ref_chrf)    
                else:
                    bleu = compute_bleu(pred, ref)
                    chrf = compute_chrf(pred, ref)

                print(f'  [{i+1:>2}/{len(pair_samples)}] ASR-BLEU={bleu:5.1f} ASR-ChrF={chrf:5.1f} RTF={rtf:.3f}')
                print(f'              pred: {pred[:80]}')
                
                if save_n > 0 and i < save_n:
                    play(s['wav'], 16000, label=f'{label}_{pair_key}_s{i+1}in.wav')
                    save_audio(s['wav'], 16000, f'{label}_{pair_key}_s{i+1}in.wav')
                    play(wav_out, 16000, label=f'{label}_{pair_key}_s{i+1}out.wav')
                    save_audio(wav_out, 16000, f'{label}_{pair_key}_s{i+1}out.wav')
                
                results.append(dict(
                    id=s['id'], src_lang=s['src_lang'], tgt_lang=s['tgt_lang'],
                    bleu=bleu, chrf=chrf, rtf=rtf, pred=pred, ref=s['ref']))
            except Exception as e:
                import traceback; traceback.print_exc()
                results.append(dict(
                    id=s['id'], src_lang=s.get('src_lang','?'), tgt_lang=s.get('tgt_lang','?'),
                    bleu=0, chrf=0, rtf=float('nan'), pred='', ref=s.get('ref','')))
    
    valid = [r for r in results if not math.isnan(r['rtf'])]
    summary = dict(
        label=label, n=len(valid),
        avg_bleu=float(np.mean([r['bleu'] for r in valid])) if valid else 0,
        avg_chrf=float(np.mean([r['chrf'] for r in valid])) if valid else 0,
        avg_rtf =float(np.mean([r['rtf']  for r in valid])) if valid else 0,
        params_M=count_params(mdl)
    )
    
    # Per-pair breakdown
    print(f'\n  === Summary by Language Pair ===')
    for pair_key in by_pair.keys():
        pair_res = [r for r in valid if f"{r['src_lang']}→{r['tgt_lang']}" == pair_key]
        if pair_res:
            avg_chrf_pair = np.mean([r['chrf'] for r in pair_res])
            avg_bleu_pair = np.mean([r['bleu'] for r in pair_res])
            print(f'  {pair_key:<12} ASR-ChrF={avg_chrf_pair:5.2f}  ASR-BLEU={avg_bleu_pair:5.2f}')
    
    print(f'\n  Overall: ASR-BLEU={summary["avg_bleu"]:.2f} ASR-ChrF={summary["avg_chrf"]:.2f}'
          f' RTF={summary["avg_rtf"]:.4f} Params={summary["params_M"]:.1f}M')
    return results, summary

# Alias for backward compatibility
run_benchmark = run_benchmark_asr


In [20]:
from transformers import SeamlessM4Tv2ForSpeechToSpeech, SeamlessM4TProcessor

try:
    HF_TOKEN = _get_secret('HF_TOKEN')
    from huggingface_hub import login
    login(HF_TOKEN)
    print('Logged into HuggingFace Hub.')
except Exception as e:
    print(f'HF login skipped: {e}')

MODEL_NAME = 'facebook/seamless-m4t-v2-large'
processor = None   # Will be set when loading any model

def load_base_model():
    global processor
    print(f'Loading processor from {MODEL_NAME}...')
    proc = SeamlessM4TProcessor.from_pretrained(MODEL_NAME)
    print(f'Loading model -- may take 5-10 min...')
    mdl = SeamlessM4Tv2ForSpeechToSpeech.from_pretrained(
        MODEL_NAME, torch_dtype=torch.float16, device_map='auto')
    mdl.eval()
    print('Model loaded.'); gpu_mem()
    processor = proc
    return mdl, proc

print('load_base_model() ready. Call it to load teacher/base model.')


Logged into HuggingFace Hub.
load_base_model() ready. Call it to load teacher/base model.


In [21]:
## Dataset loading — battle-tested from seamless-cse465v5 (Cells 24-26)
import concurrent.futures, io, soundfile as sfile, pandas as pd

# LOCAL_PARQUET_CACHE = '/kaggle/working/fleurs_parquet'
LOCAL_PARQUET_CACHE = '/kaggle/input/datasets/rayedriasat/fleurs5'
BASE_PARQUET_URL = 'https://huggingface.co/datasets/google/fleurs/resolve/refs%2Fconvert%2Fparquet'
DRIVE_FLEURS_PATH = f'{GDRIVE_ROOT}/fleurs_parquet'

def _list_parquet_urls(lang, split):
    import requests
    urls, i = [], 0
    while True:
        url = f'{BASE_PARQUET_URL}/{lang}/{split}/{i:04d}.parquet?download=true'
        try:
            r = requests.head(url, timeout=15, allow_redirects=True)
            if r.status_code == 200: urls.append(url); i += 1
            else: break
        except: break
    if not urls:
        urls = [f'{BASE_PARQUET_URL}/{lang}/{split}/0000.parquet?download=true']
        print(f'  [WARN] fallback to shard 0000 for {lang}/{split}')
    print(f'  [shards] {lang}/{split}: {len(urls)} shard(s)')
    return urls

def _download_shard(args):
    import requests
    url, dest = args
    dest = pathlib.Path(dest)
    if dest.exists() and dest.stat().st_size > 1024*1024:
        return url, True, 'cached'
    dest.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(3):
        try:
            r = requests.get(url, stream=True, timeout=120)
            r.raise_for_status()
            with open(dest, 'wb') as f:
                for chunk in r.iter_content(8*1024*1024):
                    if chunk: f.write(chunk)
            if dest.stat().st_size > 1024*1024: return url, True, 'downloaded'
            raise RuntimeError('Downloaded file too small')
        except Exception as e:
            if dest.exists(): dest.unlink()
            if attempt == 2: return url, False, str(e)
    return url, False, 'unknown'

def load_fleurs_parallel(src_lang, tgt_lang, split='train', n_workers=4):
    from datasets import Dataset
    tasks = []
    for lang in [src_lang, tgt_lang]:
        urls = _list_parquet_urls(lang, split)
        for i, url in enumerate(urls):
            dest = f'{LOCAL_PARQUET_CACHE}/{lang}/{split}_{i:04d}.parquet'
            tasks.append((url, dest))
    print(f'[Parallel] Downloading {len(tasks)} shards...')
    with concurrent.futures.ThreadPoolExecutor(max_workers=n_workers) as pool:
        for url, ok, msg in pool.map(_download_shard, tasks):
            print(f'  {"OK" if ok else "FAIL"}: {msg}')
    def _load_lang(lang):
        if lang == "": 
            return None
        files = sorted(glob.glob(f'{LOCAL_PARQUET_CACHE}/{lang}/{split}_*.parquet'))
        if not files: raise FileNotFoundError(f'No cached shards for {lang}')
        return Dataset.from_pandas(pd.read_parquet(files[0]))
    return _load_lang(src_lang), _load_lang(tgt_lang)

def push_fleurs_to_drive():
    if not ON_KAGGLE: return
    subprocess.run(f'rclone copy "{LOCAL_PARQUET_CACHE}/" "{DRIVE_FLEURS_PATH}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
                   shell=True, capture_output=True)

def load_fleurs_from_drive(src_lang, tgt_lang, split='train'):
    from datasets import Dataset
    if not ON_KAGGLE: return None, None
    if not os.path.exists(LOCAL_PARQUET_CACHE):
        r = subprocess.run(f'rclone copy "{DRIVE_FLEURS_PATH}/" "{LOCAL_PARQUET_CACHE}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
                           shell=True, capture_output=True, text=True)
        if r.returncode != 0: return None, None
    def _load_lang(lang):
        if lang == "": 
            return None
        files = sorted(glob.glob(f'{LOCAL_PARQUET_CACHE}/{lang}/{split}_*.parquet'))
        if not files: return None
        return Dataset.from_pandas(pd.concat([pd.read_parquet(f) for f in files], ignore_index=True))
    src_ds = _load_lang(src_lang); tgt_ds = _load_lang(tgt_lang)
    if src_ds and tgt_ds: print(f'[gdrive] Loaded: {len(src_ds)} src, {len(tgt_ds)} tgt')
    return src_ds, tgt_ds

def _load_wav(audio_cell):
    """Verbatim from v5 Cell 25 — handles both HF Dataset and parquet byte formats."""    
    audio = audio_cell
    if isinstance(audio, dict) and 'array' in audio:
        arr, sr = audio['array'], audio['sampling_rate']
    elif isinstance(audio, dict) and 'bytes' in audio:
        wav, sr = sfile.read(io.BytesIO(audio['bytes']))
        if wav.ndim > 1: wav = wav.mean(axis=1)
        arr = wav
    else:
        raise RuntimeError(f'Unsupported audio format: {type(audio)}')
    arr = np.array(arr, dtype=np.float32)
    if sr != 16000:
        arr = torchaudio.functional.resample(torch.tensor(arr), sr, 16000).numpy()
    return arr

print('FLEURS data loaders ready.')


FLEURS data loaders ready.


In [22]:
# from datasets import Dataset
# tasks = []
# split = 'validation'
# for lang in ['ar_eg', 'bn_in', 'cmn_hans_cn', 'en_us', 'hi_in']:
#     urls = _list_parquet_urls(lang, split)
#     for i, url in enumerate(urls):
#         dest = f'{LOCAL_PARQUET_CACHE}/{lang}/{split}_{i:04d}.parquet'
#         tasks.append((url, dest))
# print(f'[Parallel] Downloading {len(tasks)} shards...')
# with concurrent.futures.ThreadPoolExecutor(max_workers=16) as pool:
#     for url, ok, msg in pool.map(_download_shard, tasks):
#         print(f'  {"OK" if ok else "FAIL"}: {msg}')
# push_fleurs_to_drive()

In [23]:
import os, glob, torch
from datetime import datetime

def session_status():
    print('=' * 65)
    print(f'  Platform : {PLATFORM}   Time : {datetime.now():%Y-%m-%d %H:%M}')
    if os.path.exists(CKPT_DIR):
        files = [f for f in glob.glob(f'{CKPT_DIR}/**/*.pt', recursive=True) if os.path.isfile(f)]
        print(f'  Checkpoint files: {len(files)}')
        for f in sorted(files)[:20]:
            print(f'    {os.path.relpath(f,CKPT_DIR):<50} {os.path.getsize(f)/1e6:>8.1f} MB')
    if torch.cuda.is_available():
        props = torch.cuda.get_device_properties(0)
        print(f'  GPU: {torch.cuda.get_device_name(0)}  VRAM: {props.total_memory/1e9:.1f} GB')
    print('=' * 65)

if not os.path.exists(LOCAL_PARQUET_CACHE):
    r = subprocess.run(f'rclone copy "{DRIVE_FLEURS_PATH}/" "{LOCAL_PARQUET_CACHE}/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M',
                       shell=True, capture_output=True, text=True)

sync_checkpoints_from_drive()
session_status()
print('\n✓ ALL SETUP CELLS COMPLETE — proceed to phases.')


[ckpt] Syncing from rclone remote...
[ckpt] 24 file(s) available
  all_detailed_summaries_step000000.pt                        0.0 MB
  all_summaries_step000000.pt                                 0.0 MB
  phase0_benchmark_step000000.pt                              0.1 MB
  phase1_benchmark_step000000.pt                              0.1 MB
  phase2_benchmark_step000000.pt                              0.1 MB
  phase2_enc_pruning_step000000.pt                            0.0 MB
  phase3_benchmark_step000000.pt                              0.1 MB
  phase3_laco_done_step000000.pt                              0.0 MB
  phase4_benchmark_step000000.pt                              0.1 MB
  phase4_enc_pruning_step000000.pt                            0.0 MB
  phase5_benchmark_step000000.pt                              0.1 MB
  phase5_dec_pruning_step000000.pt                            0.0 MB
  phase6_6b1_step000200.pt                                  939.8 MB
  phase6_6b2_step001350.pt            

In [24]:
# ══════════════════════════════════════════════════════════════════════════════
# RAM-Efficient Parquet Streaming Dataset
# Loads audio on-demand, not during initialization
# RAM: ~4MB for 4000 samples (vs ~20GB with old approach)
# ══════════════════════════════════════════════════════════════════════════════

import pyarrow.parquet as pq

class ParquetStreamingDataset:
    """Memory-efficient dataset that streams from parquet files."""
    
    def __init__(self, parquet_cache_dir, src_lang, tgt_lang, split='train', 
                 max_samples_per_pair=500):
        self.cache_dir = pathlib.Path(parquet_cache_dir)
        self.src_lang = src_lang
        self.tgt_lang = tgt_lang
        self.split = split
        self.max_samples = max_samples_per_pair
        self.samples = []
        self._build_index()
    
    def _build_index(self):
        """Build lightweight index (metadata only, no audio)."""
        src_files = sorted(self.cache_dir.glob(f'{M4T_FLEURS_MAP.get(self.src_lang)}/{self.split}_*.parquet'))
        tgt_files = sorted(self.cache_dir.glob(f'{M4T_FLEURS_MAP.get(self.tgt_lang)}/{self.split}_*.parquet'))
        
        if not src_files or not tgt_files:
            print(f'  WARNING: No parquet files for {self.src_lang}/{self.tgt_lang}')
            return
        
        # Read only ID columns (fast, <1MB RAM)
        src_ids = []
        for f in src_files:
            df = pd.read_parquet(f, columns=['id'])
            src_ids.extend([(str(f), idx, row_id) for idx, row_id in enumerate(df['id'])])
        
        tgt_ids = []
        for f in tgt_files:
            df = pd.read_parquet(f, columns=['id', 'transcription'])
            df = df[df['transcription'].str.strip().str.len() > 0]
            tgt_ids.extend([(str(f), idx, row_id, trans) 
                           for idx, (row_id, trans) in enumerate(zip(df['id'], df['transcription']))])
        
        # Create lookup dicts
        src_lookup = {row_id: (f, idx) for f, idx, row_id in src_ids}
        tgt_lookup = {row_id: (f, idx, trans) for f, idx, row_id, trans in tgt_ids}
        
        # Find matching IDs
        common_ids = set(src_lookup.keys()) & set(tgt_lookup.keys())
        
        # Build sample index (metadata only)
        for sample_id in list(common_ids)[:self.max_samples]:
            src_file, src_idx = src_lookup[sample_id]
            tgt_file, tgt_idx, tgt_text = tgt_lookup[sample_id]
            
            self.samples.append({
                'id': f"{self.src_lang}2{self.tgt_lang}_{sample_id}",
                'src_lang': self.src_lang,
                'tgt_lang': self.tgt_lang,
                'ref': tgt_text,
                '_src_file': src_file,
                '_src_idx': src_idx,
            })
        
        print(f'  Indexed {len(self.samples)} samples from {self.src_lang}→{self.tgt_lang}')
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        """Get sample with audio loaded on-demand."""
        sample = self.samples[idx].copy()
        
        # Load audio only when accessed
        if '_src_file' in sample:
            audio = self._load_audio_from_parquet(
                sample['_src_file'], 
                sample['_src_idx']
            )
            sample['wav'] = audio
            del sample['_src_file']
            del sample['_src_idx']
        
        return sample
    
    def _load_audio_from_parquet(self, parquet_file, row_idx):
        """Load single audio sample from parquet file."""
        table = pq.read_table(parquet_file, columns=['audio'])
        audio_cell = table.to_pandas().iloc[row_idx]['audio']
        return _load_wav(audio_cell)


class MultilingualStreamingDataset:
    """Combines multiple language pairs into a single streaming dataset."""
    
    def __init__(self, parquet_cache_dir, lang_pairs, split='train', 
                 max_samples_per_pair=25):
        self.datasets = []
        
        for src_lang, tgt_lang in lang_pairs:
            ds = ParquetStreamingDataset(
                parquet_cache_dir, src_lang, tgt_lang, split, max_samples_per_pair
            )
            if len(ds) > 0:
                self.datasets.append(ds)
        
        # Build flat index
        self.index = []
        for ds_idx, ds in enumerate(self.datasets):
            for sample_idx in range(len(ds)):
                self.index.append((ds_idx, sample_idx))
        
        print(f'\n✓ Multilingual dataset ready: {len(self.index)} total samples')
        print(f'  RAM usage: ~{len(self.index) * 0.001:.1f} MB (metadata only)')
    
    def __len__(self):
        return len(self.index)
    
    def __getitem__(self, idx):
        """Get sample from appropriate sub-dataset."""
        if isinstance(idx, slice):
            indices = range(*idx.indices(len(self)))
            return [self[i] for i in indices]
        ds_idx, sample_idx = self.index[idx]
        return self.datasets[ds_idx][sample_idx]
    
    def __iter__(self):
        """Allow iteration."""
        for i in range(len(self)):
            yield self[i]

print('✓ Streaming dataset classes ready.')

✓ Streaming dataset classes ready.


In [25]:
# ── Load Multilingual Eval Samples: En→X and X→En (all 5 languages) ─────────
# PLAN.md Section 5: 5 languages — EN, BN, ZH, AR, HI
N_EVAL_PER_PAIR = 25
EVAL_LANG_PAIRS = [
    ('eng', 'ben'), ('ben', 'eng'),  # English ↔ Bengali
    ('eng', 'cmn'), ('cmn', 'eng'),  # English ↔ Mandarin
    ('eng', 'arb'), ('arb', 'eng'),  # English ↔ Arabic
    ('eng', 'hin'), ('hin', 'eng'),  # English ↔ Hindi
]

In [26]:
# ── Load Multilingual Eval Samples: En→X and X→En (all 5 languages) ──────────
# STREAMING VERSION: Only loads audio when accessed
# RAM: ~200KB for 200 samples (vs ~1GB with old approach)

print('Loading evaluation samples (streaming mode)...')
eval_samples = MultilingualStreamingDataset(
    parquet_cache_dir=LOCAL_PARQUET_CACHE,
    lang_pairs=EVAL_LANG_PAIRS,
    split='test',
    max_samples_per_pair=N_EVAL_PER_PAIR
)

print(f'\n✓ Loaded {len(eval_samples)} multilingual eval samples')
print(f'  Language pairs: {len(EVAL_LANG_PAIRS)}')
print(f'  RAM usage: ~{len(eval_samples) * 0.001:.1f} MB (metadata only)')

# Test: Load one sample to verify it works
test_sample = eval_samples[26]
print(f'\n✓ Test sample loaded:')
print(f'  ID: {test_sample["id"]}')
print(f'  Audio shape: {test_sample["wav"].shape}')
print(f'  Reference: {test_sample["ref"][:50]}...')

play(test_sample["wav"], 16000, label='hello.wav')

Loading evaluation samples (streaming mode)...
  Indexed 25 samples from eng→ben
  Indexed 25 samples from ben→eng
  Indexed 25 samples from eng→cmn
  Indexed 25 samples from cmn→eng
  Indexed 25 samples from eng→arb
  Indexed 25 samples from arb→eng
  Indexed 25 samples from eng→hin
  Indexed 25 samples from hin→eng

✓ Multilingual dataset ready: 200 total samples
  RAM usage: ~0.2 MB (metadata only)

✓ Loaded 200 multilingual eval samples
  Language pairs: 8
  RAM usage: ~0.2 MB (metadata only)

✓ Test sample loaded:
  ID: ben2eng_1661
  Audio shape: (187200,)
  Reference: he did not set a figure for the cuts saying they w...
  hello.wav  (11.7s | sr=16000)


In [27]:
# ── Load Multilingual Training Samples: En→X and X→En (all 5 languages) ─────
N_TRAIN_PER_PAIR = 1200  # 500 samples per direction = 4000 total

In [28]:
# ── Load Multilingual Training Samples: En→X and X→En (all 5 languages) ──────
# STREAMING VERSION: Only loads audio when accessed
# RAM: ~4MB for 4000 samples (vs ~20GB with old approach)

print('Loading training samples (streaming mode)...')
ft_samples = MultilingualStreamingDataset(
    parquet_cache_dir=LOCAL_PARQUET_CACHE,
    lang_pairs=EVAL_LANG_PAIRS,
    split='train',
    max_samples_per_pair=N_TRAIN_PER_PAIR
)

print(f'\n✓ Loaded {len(ft_samples)} multilingual training samples')
print(f'  Language pairs: {len(EVAL_LANG_PAIRS)}')
print(f'  RAM usage: ~{len(ft_samples) * 0.001:.1f} MB (metadata only)')
print(f'  RAM saved: ~{len(ft_samples) * 5:.0f} MB (would be with old approach)')

# Summary by language pair
print('\nSamples per language pair:')
pair_counts = {}
for i in range(len(ft_samples)):
    sample_meta = ft_samples.datasets[ft_samples.index[i][0]].samples[ft_samples.index[i][1]]
    pair = f"{sample_meta['src_lang']}→{sample_meta['tgt_lang']}"
    pair_counts[pair] = pair_counts.get(pair, 0) + 1

for pair, count in sorted(pair_counts.items()):
    print(f'  {pair}: {count}')

Loading training samples (streaming mode)...
  Indexed 1200 samples from eng→ben
  Indexed 1200 samples from ben→eng
  Indexed 1200 samples from eng→cmn
  Indexed 1200 samples from cmn→eng
  Indexed 1200 samples from eng→arb
  Indexed 1200 samples from arb→eng
  Indexed 1200 samples from eng→hin
  Indexed 1200 samples from hin→eng

✓ Multilingual dataset ready: 9600 total samples
  RAM usage: ~9.6 MB (metadata only)

✓ Loaded 9600 multilingual training samples
  Language pairs: 8
  RAM usage: ~9.6 MB (metadata only)
  RAM saved: ~48000 MB (would be with old approach)

Samples per language pair:
  arb→eng: 1200
  ben→eng: 1200
  cmn→eng: 1200
  eng→arb: 1200
  eng→ben: 1200
  eng→cmn: 1200
  eng→hin: 1200
  hin→eng: 1200


In [29]:
# ── Multilingual eval samples now integrated into eval_samples ──────────────
# All 5 languages (EN, BN, HI, ZH, AR) with bidirectional pairs are loaded above
print(f'Multilingual eval ready: {len(eval_samples)} samples across {len(EVAL_LANG_PAIRS)} pairs')
print(f'Language pairs: {EVAL_LANG_PAIRS}')

print(f'\n✓ Loaded {len(ft_samples)} multilingual training samples across {len(EVAL_LANG_PAIRS)} pairs')


Multilingual eval ready: 200 samples across 8 pairs
Language pairs: [('eng', 'ben'), ('ben', 'eng'), ('eng', 'cmn'), ('cmn', 'eng'), ('eng', 'arb'), ('arb', 'eng'), ('eng', 'hin'), ('hin', 'eng')]

✓ Loaded 9600 multilingual training samples across 8 pairs


# PHASE 6: LoRA + Native T2U Recovery

This Phase 6 block fully replaces the earlier broken LoRA/DoRA cells.

Design rules used here:

- exact module names from `AAA/modeling_seamless_m4t_v2.py`
- LoRA on `speech_encoder + text_decoder`
- full native `t2u_model` recovery for the NAR text-to-unit path
- offline teacher cache
- strict 2-GPU separation when available
- fp16, gradient checkpointing, micro-batch 1, short-audio caps

Run the cells in order. If Stage 6C still OOMs, set `PHASE6_T2U_TRAIN_MODE = "selective"` in the setup cell and rerun Stage 6C.


In [ ]:
import gc
import glob
import math
import os
import random
import time
from collections import defaultdict, OrderedDict

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from peft import LoraConfig, get_peft_model

PHASE6_MODEL_NAME = 'phase6_lora_t2u_merged'
PHASE6_BENCHMARK_NAME = 'phase6_lora_t2u_benchmark'
PHASE6_CACHE_PREFIX = 'phase6_teacher_cache'

MICRO_BATCH    = 1
GRAD_ACCUM     = 8
MAX_AUDIO_SEC_B1 = 20
MAX_AUDIO_SEC_B2 = 20
MAX_AUDIO_SEC_C  = 12
MAX_AUDIO_SEC_D  = 14
MAX_GRAD_NORM    = 1.0
WARMUP_RATIO     = 0.10

# ── Logging / eval / checkpoint cadence ───────────────────────────────────────
# LOG_EVERY   : print a loss line every N optimizer steps (keep this low — you want feedback)
# EVAL_EVERY  : run phase6_quick_eval every N optimizer steps
# SAVE_EVERY  : save a checkpoint every N optimizer steps
# These are absolute step counts. Eval and Save are also clamped so they never
# fire more often than LOG_EVERY regardless of what you set.
LOG_EVERY  = 10    # print every 25 opt steps  → ~200 fwd passes between prints
EVAL_EVERY = 50   # quick eval every 100 steps → enough to track quality
SAVE_EVERY = 50   # checkpoint every 200 steps → you lose at most 200 steps on crash


# ── All STEPS values are OPTIMIZER STEPS (gradient updates), NOT micro-steps.
# ── Micro-steps per run = STEPS × GRAD_ACCUM  (e.g. 1200 × 8 = 9600 fwd passes)
# ── Warmup covers the first WARMUP_RATIO × STEPS optimizer steps as intended.
# ── To extend a run, increase the value here and re-run — auto-resume handles the rest.
STAGE6B1_STEPS   = 400    # ~3,200 fwd passes  — LoRA warmup, converges fast
STAGE6B2_STEPS   = 900    # ~7,200 fwd passes  — joint LoRA recovery
STAGE6C_STEPS    = 700    # ~5,600 fwd passes  — T2U distillation
STAGE6D_STEPS    = 350    # ~2,800 fwd passes  — polish, keep short
STAGE6D_ENABLED  = True

PHASE6_TEXT_KD_PROB   = 0.2
PHASE6_T2U_TRAIN_MODE = 'full'


student_device = torch.device('cuda:0')
teacher_device = torch.device('cuda:1' if torch.cuda.device_count() > 1 else 'cuda:0')

phase6_logs = {'6b1': [], '6b2': [], '6c': [], '6d': []}
phase6_eval_history = []
phase6_cache_manifest = {}
phase6_cache_index = {}
phase6_cache_keys_by_pair = defaultdict(list)
phase6_sample_lookup = {}
phase6_shard_cache = OrderedDict()

PHASE6_CACHE_SHARD_SIZE = 2048
PHASE6_CACHE_SYNC_PARTS = 4
PHASE6_SHARD_LRU_LIMIT = 16
PHASE6_MAX_TEACHER_UNIT_TOKENS = 2048
PHASE6_MIN_TEACHER_UNIT_TOKENS = 3

phase6_cache_stats = {
    'cached_ok': 0,
    'skipped_empty_text': 0,
    'skipped_short_units': 0,
    'skipped_long_units': 0,
    'skipped_errors': 0,
}


class _NoopVocoder(nn.Module):
    def __init__(self, device):
        super().__init__()
        self.device = torch.device(device)

    def forward(self, *args, **kwargs):
        input_ids = kwargs.get('input_ids', args[0] if args else None)
        if input_ids is None:
            raise RuntimeError('Noop vocoder expected input_ids.')
        batch = input_ids.shape[0]
        waveform = torch.zeros(batch, 1, device=self.device)
        lengths = torch.ones(batch, dtype=torch.int32, device=self.device)
        return waveform, lengths


def safe_gc():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def move_model_to_device(mdl, device):
    if not torch.cuda.is_available():
        return mdl
    try:
        from accelerate.hooks import remove_hook_from_submodules
        remove_hook_from_submodules(mdl)
    except Exception:
        pass
    return mdl.to(device)


def maybe_enable_gradient_checkpointing(module, label):
    fn = getattr(module, 'gradient_checkpointing_enable', None)
    if callable(fn):
        fn()
        print(f'  gradient checkpointing enabled: {label}')


def disable_generation_cache(mdl):
    if hasattr(mdl, 'config'):
        mdl.config.use_cache = False
    if hasattr(mdl, 'text_decoder') and hasattr(mdl.text_decoder, 'config'):
        mdl.text_decoder.config.use_cache = False


def phase6_sample_key(sample_or_entry):
    sample_id = sample_or_entry.get('sample_id', sample_or_entry.get('id'))
    return f"{sample_or_entry['src_lang']}__{sample_or_entry['tgt_lang']}__{sample_id}"


def build_sample_lookup(streaming_ds):
    lookup = {}
    for outer_idx, (ds_idx, sample_idx) in enumerate(streaming_ds.index):
        meta = streaming_ds.datasets[ds_idx].samples[sample_idx]
        key = f"{meta['src_lang']}__{meta['tgt_lang']}__{meta['id']}"
        if key in lookup:
            raise RuntimeError(f'Duplicate Phase 6 sample key found: {key}')
        lookup[key] = outer_idx
    return lookup


def get_submodule_strict(root, dotted_name: str):
    cur = root
    for part in dotted_name.split('.'):
        if part.isdigit():
            cur = cur[int(part)]
        else:
            if not hasattr(cur, part):
                raise AttributeError(f"Missing submodule part '{part}' in '{dotted_name}'")
            cur = getattr(cur, part)
    return cur


def assert_linear_targets_exist(model, target_names):
    bad = []
    for name in target_names:
        mod = get_submodule_strict(model, name)
        if not isinstance(mod, nn.Linear):
            bad.append((name, type(mod).__name__))
    if bad:
        msg = '\n'.join([f'{name} -> {kind}' for name, kind in bad])
        raise TypeError(f'Non-linear LoRA targets found:\n{msg}')


def build_speech_encoder_lora_targets(model_student):
    n = len(model_student.speech_encoder.encoder.layers)
    targets = [
        'feature_projection.projection',
        'intermediate_ffn.intermediate_dense',
        'intermediate_ffn.output_dense',
    ]
    for i in range(n):
        prefix = f'encoder.layers.{i}'
        targets += [
            f'{prefix}.self_attn.linear_q',
            f'{prefix}.self_attn.linear_k',
            f'{prefix}.self_attn.linear_v',
            f'{prefix}.self_attn.linear_out',
            f'{prefix}.ffn1.intermediate_dense',
            f'{prefix}.ffn1.output_dense',
            f'{prefix}.ffn2.intermediate_dense',
            f'{prefix}.ffn2.output_dense',
        ]
    assert_linear_targets_exist(model_student.speech_encoder, targets)
    return targets


def build_text_decoder_lora_targets(model_student):
    n = len(model_student.text_decoder.layers)
    targets = []
    for i in range(n):
        prefix = f'layers.{i}'
        targets += [
            f'{prefix}.self_attn.q_proj',
            f'{prefix}.self_attn.k_proj',
            f'{prefix}.self_attn.v_proj',
            f'{prefix}.self_attn.out_proj',
            f'{prefix}.cross_attention.q_proj',
            f'{prefix}.cross_attention.k_proj',
            f'{prefix}.cross_attention.v_proj',
            f'{prefix}.cross_attention.out_proj',
            f'{prefix}.ffn.fc1',
            f'{prefix}.ffn.fc2',
        ]
    assert_linear_targets_exist(model_student.text_decoder, targets)
    return targets


def make_lora_config(target_modules, r, alpha, dropout=0.05):
    return LoraConfig(
        target_modules=target_modules,
        r=r,
        lora_alpha=alpha,
        lora_dropout=dropout,
        bias='none',
        use_rslora=True,
    )


def wrap_with_lora_if_needed(module, config, label):
    if hasattr(module, 'peft_config'):
        print(f'{label} already has LoRA adapters attached.')
        return module
    wrapped = get_peft_model(module, config)
    print(f'{label} LoRA attached.')
    wrapped.print_trainable_parameters()
    return wrapped


def freeze_all_student():
    for p in model_student.parameters():
        p.requires_grad_(False)


def enable_lora_params(module, label):
    found = 0
    for name, p in module.named_parameters():
        if 'lora_' in name:
            p.requires_grad_(True)
            found += p.numel()
    if found == 0:
        raise RuntimeError(f'No LoRA parameters found in {label}.')
    print(f'  enabled LoRA params for {label}: {found/1e6:.2f}M')


def mark_t2u_trainable_full():
    for p in model_student.parameters():
        p.requires_grad_(False)
    for p in model_student.t2u_model.parameters():
        p.requires_grad_(True)
    print('  T2U mode: full native training')


def mark_t2u_selective_trainable():
    for p in model_student.parameters():
        p.requires_grad_(False)
    t2u = model_student.t2u_model
    for name, p in t2u.named_parameters():
        if (
            name.startswith('model.decoder.layers.')
            or name.startswith('model.decoder.duration_predictor.')
            or name in {
                'model.decoder.pos_emb_alpha_char',
                'model.decoder.pos_emb_alpha',
                'lm_head.weight',
            }
        ):
            p.requires_grad_(True)
    print('  T2U mode: selective decoder + duration predictor')


def count_trainable_params(module):
    return sum(p.numel() for p in module.parameters() if p.requires_grad) / 1e6


def trainable_named_params(module):
    return [p for p in module.parameters() if p.requires_grad]


def make_cosine_scheduler(optimizer, total_steps, warmup_ratio=WARMUP_RATIO, eta_min_ratio=0.05):
    warmup_steps = max(1, int(total_steps * warmup_ratio))

    def lr_lambda(step):
        if step < warmup_steps:
            return float(step + 1) / float(warmup_steps)
        progress = (step - warmup_steps) / max(1, total_steps - warmup_steps)
        cosine_decay = 0.5 * (1.0 + math.cos(math.pi * progress))
        # Floor at eta_min_ratio so LR never reaches zero
        return eta_min_ratio + (1.0 - eta_min_ratio) * cosine_decay

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

### ADD THIS: Cyclic LR scheduler for resume extensions
def make_cyclic_scheduler(optimizer, base_lr=1.5e-5, max_lr=8e-5,
                          cycle_steps=75, warmup_within_cycle=8):
    """
    Triangular cyclic LR. The optimizer's param_group['lr'] MUST already be
    set to base_lr before this scheduler is constructed — LambdaLR snapshots
    base_lrs at __init__ time and the lambda multiplies from that base.

    Lambda output range: 1.0 (base_lr) → ratio (max_lr) → 1.0, repeating.
    """
    if base_lr <= 0:
        raise ValueError(f'base_lr must be > 0, got {base_lr}')
    ratio = max_lr / base_lr  # e.g. 8e-5 / 1.5e-5 ≈ 5.33

    def lr_lambda(step):
        cycle_pos = step % cycle_steps
        if cycle_pos < warmup_within_cycle:
            # Ramp up: 1.0 → ratio
            progress = cycle_pos / max(1, warmup_within_cycle)
            return 1.0 + (ratio - 1.0) * progress
        else:
            # Ramp down: ratio → 1.0
            progress = (cycle_pos - warmup_within_cycle) / max(1, cycle_steps - warmup_within_cycle)
            return ratio - (ratio - 1.0) * progress

    return torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

def phase6_prepare_audio_inputs(sample, device):
    inputs = processor(
        audio=sample['wav'],
        sampling_rate=16000,
        return_tensors='pt',
    )
    return {k: v.to(device) for k, v in inputs.items()}


def phase6_quick_eval(tag, max_samples=16):
    model_student.eval()
    text_score, asr_score = quick_eval_chrf(model_student, eval_samples, max_samples=max_samples)
    phase6_eval_history.append({'tag': tag, 'text_chrf': text_score, 'asr_chrf': asr_score})
    print(f'  [{tag}] quick Text-ChrF: {text_score:.2f} | ASR-ChrF: {asr_score:.2f}')
    model_student.train()
    return text_score, asr_score


def phase6_raise_oom(stage_name, step_idx, max_audio_sec, extra=''):
    msg = (
        f'{stage_name} hit CUDA OOM at step {step_idx}. '
        f'Reduce max audio below {max_audio_sec}s'
    )
    if extra:
        msg += f' or {extra}'
    raise RuntimeError(msg)


def phase6_get_cache_entry(sample_key):
    if sample_key not in phase6_cache_index:
        raise KeyError(f'Missing cache index for {sample_key}')

    ref = phase6_cache_index[sample_key]
    shard_idx = ref['shard_idx']
    split_name = ref['split']
    shard_key = f'{split_name}:{shard_idx}'

    if shard_key in phase6_shard_cache:
        entries = phase6_shard_cache.pop(shard_key)
        phase6_shard_cache[shard_key] = entries
    else:
        shard_path = os.path.join(
            CKPT_DIR,
            f"{phase6_cache_checkpoint_name(split_name)}_step{shard_idx:06d}.pt",
        )
        if not os.path.exists(shard_path):
            raise RuntimeError(f'Cache shard missing locally: {shard_path}')
        shard_blob = torch.load(shard_path, map_location='cpu', weights_only=False)
        entries = shard_blob.get('entries', [])
        phase6_shard_cache[shard_key] = entries
        while len(phase6_shard_cache) > PHASE6_SHARD_LRU_LIMIT:
            phase6_shard_cache.popitem(last=False)

    return entries[ref['offset']]


def phase6_pick_training_pair(max_audio_sec, balanced=True):
    if not phase6_cache_keys_by_pair:
        raise RuntimeError('Teacher cache index is not loaded. Run the Phase 6A cache cell first.')

    pair_keys = list(phase6_cache_keys_by_pair.keys())
    all_keys = list(phase6_cache_index.keys())

    for _ in range(512):
        pair = random.choice(pair_keys) if balanced else None
        candidates = phase6_cache_keys_by_pair[pair] if balanced else all_keys
        cache_key = random.choice(candidates)
        ref = phase6_cache_index[cache_key]
        if ref['audio_len_s'] <= max_audio_sec:
            sample_idx = phase6_sample_lookup[cache_key]
            sample = ft_samples[sample_idx]
            entry = phase6_get_cache_entry(cache_key)
            return sample, entry

    raise RuntimeError(
        f'Could not find a training sample under {max_audio_sec}s. '
        'Lower the dataset cap or increase the per-stage audio budget.'
    )


def teacher_generate_tokens(model_teacher, inputs, tgt_lang):
    original_vocoder = model_teacher.vocoder
    teacher_dev = next(model_teacher.parameters()).device
    model_teacher.vocoder = _NoopVocoder(teacher_dev)
    try:
        with torch.no_grad():
            return model_teacher.generate(
                **inputs,
                tgt_lang=tgt_lang,
                return_intermediate_token_ids=True,
                text_num_beams=4,
                text_max_new_tokens=256,
                speech_do_sample=False,
            )
    finally:
        model_teacher.vocoder = original_vocoder


def _compute_new_attention_mask(hidden_states: torch.Tensor, seq_lens: torch.Tensor):
    """Mirrors SeamlessM4Tv2ForSpeechToSpeech._compute_new_attention_mask."""
    batch_size, mask_seq_len = hidden_states.shape[:2]
    indices = torch.arange(mask_seq_len, device=seq_lens.device).expand(batch_size, -1)
    bool_mask = indices >= seq_lens.unsqueeze(1).expand(-1, mask_seq_len)
    mask = hidden_states.new_ones((batch_size, mask_seq_len))
    mask = mask.masked_fill(bool_mask, 0)
    return mask


# ── FIXED: build_t2u_conditioning uses the GIVEN model's own decoder ──────────
# The critical change: no more .detach() on the returned embeds when grad is needed.
# Caller controls whether to wrap in torch.no_grad().

def build_t2u_conditioning_from_sequences(model, input_features, attention_mask, text_sequences):
    """
    Build T2U conditioning from a model using forced text_sequences.
    Returns dict of all T2U model inputs.
    This matches the internal path in SeamlessM4Tv2ForSpeechToSpeech.generate().
    """
    # Step 1: speech encoder
    enc_out = model.speech_encoder(
        input_features=input_features,
        attention_mask=attention_mask,
        output_attentions=False,
        output_hidden_states=False,
        return_dict=True,
    )
    enc = enc_out.last_hidden_state  # [B, T_enc, D]

    # Step 2: build encoder attention mask (sub-sampled)
    encoder_attention_mask = None
    if attention_mask is not None:
        sub_lengths = model._compute_sub_sample_lengths_from_attention_mask(
            attention_mask
        ).to(enc.device)
        encoder_attention_mask = _compute_new_attention_mask(enc, sub_lengths)

    # Step 3: forced text decoder pass — gives T2U input embeddings
    # text_sequences[:, :-1] = shift right (standard teacher forcing)
    dec_out = model.text_decoder(
        input_ids=text_sequences[:, :-1],
        encoder_hidden_states=enc,
        encoder_attention_mask=encoder_attention_mask,
        use_cache=False,
        output_attentions=False,
        output_hidden_states=False,
        return_dict=True,
    )
    t2u_input_embeds = dec_out.last_hidden_state  # [B, T_txt-1, D]

    # Step 4: build char inputs (mirrors _prepare_text_to_unit_model_kwargs)
    pad_token_id = model.generation_config.pad_token_id
    eos_token_id = model.generation_config.eos_token_id

    # text_sequences[:, 2:-1] = skip BOS + lang token, skip EOS
    t2u_input_ids = text_sequences[:, 2:-1].clone()
    t2u_input_ids = torch.masked_fill(
        t2u_input_ids, t2u_input_ids == eos_token_id, pad_token_id
    )

    t2u_subwords = model._indices_to_subwords(t2u_input_ids)
    t2u_char_count_per_id = model._count_character_length_in_subword(
        t2u_input_ids,
        t2u_subwords,
        pad_token_id=pad_token_id,
    )
    pad_zero = t2u_char_count_per_id.new_zeros((t2u_char_count_per_id.shape[0], 1))
    t2u_char_count_per_id = torch.cat([pad_zero, t2u_char_count_per_id, pad_zero], dim=1)

    t2u_char_input_ids = model._get_char_input_ids(
        t2u_input_ids,
        t2u_subwords,
        t2u_char_count_per_id,
        pad_token_id=pad_token_id,
    )

    # Step 5: build T2U attention mask from text sequence lengths
    seq_lens = (text_sequences[:, :-1] != pad_token_id).int().sum(1)
    t2u_attention_mask = _compute_new_attention_mask(t2u_input_embeds, seq_lens)

    return {
        "encoder_hidden_states": enc,
        "encoder_attention_mask": encoder_attention_mask,
        "t2u_input_embeds": t2u_input_embeds,
        "t2u_attention_mask": t2u_attention_mask,
        "t2u_char_input_ids": t2u_char_input_ids,
        "t2u_char_count_per_id": t2u_char_count_per_id,
    }


# ── FIXED: loss function uses .logits, not .last_hidden_state ─────────────────

def t2u_nll_loss_from_cache(student_t2u_out, teacher_unit_ids, pad_unit_id=0):
    """
    PRIMARY LOSS: NLL of student T2U logits against cached teacher unit IDs.

    This mirrors CalcLoss in the official trainer.py exactly.
    teacher_unit_ids: [B, T_units] — from Phase 6A cache, teacher's decoded unit sequence.
    student_t2u_out.logits: [B, T_student, unit_vocab_size]

    Handles length mismatch by truncating to the shorter sequence.
    """
    # FIXED: use .logits (unit vocabulary space), NOT .last_hidden_state
    student_logits = student_t2u_out.logits  # [B, T_s, V]
    B, T_s, V = student_logits.shape
    T_t = teacher_unit_ids.shape[1]
    L = min(T_s, T_t)

    if L < 2:
        return student_logits.new_zeros(())

    s_logits = student_logits[:, :L, :].reshape(-1, V)
    t_labels = teacher_unit_ids[:, :L].reshape(-1).to(student_logits.device)

    # Mask padding
    mask = t_labels != pad_unit_id
    if mask.sum() == 0:
        return student_logits.new_zeros(())

    loss = F.cross_entropy(s_logits[mask], t_labels[mask])
    return loss


def t2u_kd_loss_from_logits(student_t2u_out, teacher_t2u_out, temperature=2.0):
    """
    SECONDARY LOSS: KL divergence between teacher and student T2U logits.
    FIXED: reads .logits (not .last_hidden_state) from both outputs.
    Only used when teacher is on GPU1 during step. Optional supplement to NLL.
    """
    # FIXED: .logits, not .last_hidden_state
    student_logits = student_t2u_out.logits  # [B, T_s, V]
    teacher_logits = teacher_t2u_out.logits  # [B, T_t, V]

    student_mask = student_t2u_out.padding_mask.bool()  # [B, T_s]
    teacher_mask = teacher_t2u_out.padding_mask.bool()  # [B, T_t]

    student_len = student_mask.sum(1).long()
    teacher_len = teacher_mask.sum(1).long()
    common_len  = torch.minimum(student_len, teacher_len)

    total_kl = student_logits.new_zeros(())
    valid = 0

    for b in range(student_logits.size(0)):
        L = int(common_len[b].item())
        if L < 2:
            continue
        s = student_logits[b, :L]
        t = teacher_logits[b, :L].to(s.device)

        total_kl = total_kl + F.kl_div(
            F.log_softmax(s / temperature, dim=-1),
            F.softmax(t / temperature, dim=-1),
            reduction="batchmean",
        ) * (temperature ** 2)
        valid += 1

    if valid == 0:
        return student_logits.new_zeros(())
    return total_kl / valid


def t2u_length_loss_normalized(student_t2u_out, teacher_unit_ids, max_len=1024.0):
    """
    FIXED length loss: normalized by max_len so it stays in [0, 1] range.
    Previously was raw token-count Smooth-L1, which hit 227.5 and dominated early gradients.
    """
    student_mask = student_t2u_out.padding_mask.bool()  # [B, T_s]
    student_len  = student_mask.sum(1).float()
    teacher_len  = (teacher_unit_ids != 0).sum(1).float().to(student_len.device)

    # Normalize so loss is in [0,1] range
    return F.smooth_l1_loss(
        student_len / max_len,
        teacher_len / max_len,
    )

# ── CORRECTED Phase 6C step function ─────────────────────────────────────────

def phase6c_step_corrected(
    model_student,
    model_teacher,
    sample,
    cache_entry,
    student_device,
    teacher_device,
    autocast_dtype=torch.float16,
    use_kd_supplement=True,
):
    """
    Corrected Phase 6C training step.

    Key fixes vs. the broken mini-test:
    1. Student conditioning uses student's OWN text_decoder (no cross-model pollution)
    2. Loss uses .logits not .last_hidden_state
    3. Primary loss is NLL against cached teacher unit IDs (offline, stable)
    4. KD is supplementary only, weighted at 0.20
    5. Length loss is normalized, weighted at 0.02 (not 0.10)
    6. Student conditioning is NOT detached from the t2u_model forward — only
       the speech_encoder and text_decoder (which are frozen) are no_grad'd
       via freeze, not via detach, so the t2u_model still receives valid grad signals
       through the embed input.
    """
    teacher_text_sequences = cache_entry["teacher_text_sequences"].unsqueeze(0)
    teacher_unit_ids       = cache_entry["teacher_unit_sequences"].unsqueeze(0)  # [1, T_units]

    audio_inputs_student = phase6_prepare_audio_inputs(sample, student_device)

    # ── Student conditioning: student's OWN frozen encoder + decoder ──────────
    # speech_encoder and text_decoder are frozen (requires_grad=False),
    # so this runs without storing activations for those modules.
    # Do NOT wrap in torch.no_grad() — t2u_model needs grad through inputs_embeds.
    with torch.cuda.amp.autocast(dtype=autocast_dtype):
        student_cond = build_t2u_conditioning_from_sequences(
            model_student,
            input_features=audio_inputs_student["input_features"],
            attention_mask=audio_inputs_student.get("attention_mask"),
            text_sequences=teacher_text_sequences.to(student_device),
        )

        # ── Student T2U forward ────────────────────────────────────────────────
        student_t2u = model_student.t2u_model(
            inputs_embeds=student_cond["t2u_input_embeds"],
            attention_mask=student_cond["t2u_attention_mask"],
            char_input_ids=student_cond["t2u_char_input_ids"],
            char_count_per_id=student_cond["t2u_char_count_per_id"],
            output_attentions=False,
            output_hidden_states=False,
            return_dict=True,
        )

        # ── PRIMARY LOSS: NLL against cached teacher unit IDs ─────────────────
        # FIXED: uses .logits (unit vocab space), not .last_hidden_state
        loss_nll = t2u_nll_loss_from_cache(
            student_t2u,
            teacher_unit_ids.to(student_device),
        )

        # ── SECONDARY LOSS: length normalization ──────────────────────────────
        # FIXED: normalized, so it stays bounded [0, 1]
        loss_len = t2u_length_loss_normalized(
            student_t2u,
            teacher_unit_ids.to(student_device),
        )

    # ── OPTIONAL SUPPLEMENT: KD from live teacher (GPU1) ──────────────────────
    # Only adds signal if teacher is loaded; skip if VRAM is tight.
    loss_kd = None
    if use_kd_supplement and model_teacher is not None:
        audio_inputs_teacher = {k: v.to(teacher_device) for k, v in audio_inputs_student.items()}
        with torch.no_grad():
            teacher_cond = build_t2u_conditioning_from_sequences(
                model_teacher,
                input_features=audio_inputs_teacher["input_features"],
                attention_mask=audio_inputs_teacher.get("attention_mask"),
                text_sequences=teacher_text_sequences.to(teacher_device),
            )
            with torch.cuda.amp.autocast(dtype=autocast_dtype):
                teacher_t2u = model_teacher.t2u_model(
                    inputs_embeds=teacher_cond["t2u_input_embeds"],
                    attention_mask=teacher_cond["t2u_attention_mask"],
                    char_input_ids=teacher_cond["t2u_char_input_ids"],
                    char_count_per_id=teacher_cond["t2u_char_count_per_id"],
                    output_attentions=False,
                    output_hidden_states=False,
                    return_dict=True,
                )
        # FIXED: KD loss uses .logits not .last_hidden_state
        with torch.cuda.amp.autocast(dtype=autocast_dtype):
            loss_kd = t2u_kd_loss_from_logits(student_t2u, teacher_t2u)

    # ── Combined loss ─────────────────────────────────────────────────────────
    # Primary: NLL (0.78) — stable, matches official trainer.py approach
    # KD: soft (0.20) — optional, supplements NLL
    # Length: (0.02) — light regularizer, normalized so it can't dominate
    if loss_kd is not None:
        loss = 0.78 * loss_nll + 0.20 * loss_kd + 0.02 * loss_len
    else:
        loss = 0.98 * loss_nll + 0.02 * loss_len

    metrics = {
        "nll": loss_nll.item(),
        "kd":  loss_kd.item() if loss_kd is not None else 0.0,
        "len": loss_len.item(),
    }
    return loss, metrics


# ── CORRECTED main training loop for Phase 6C ────────────────────────────────

def run_phase6c_corrected(
    model_student,
    model_teacher,
    student_device,
    teacher_device,
    phase6_pick_training_pair,
    phase6_quick_eval,
    phase6_cache_index,
    steps=700,
    grad_accum=8,
    max_audio_sec=12,
    base_lr=8e-5,
    dur_lr=1e-4,
    log_every=10,
    eval_every=50,
    save_every=50,
    ckpt_dir="checkpoints",
    autocast_dtype=torch.float16,
    use_kd_supplement=True,
):
    import os, math

    # Freeze everything except t2u_model
    for p in model_student.parameters():
        p.requires_grad_(False)
    for p in model_student.t2u_model.parameters():
        p.requires_grad_(True)

    # Cast t2u_model trainable params to FP32 for stable grad scaler
    n_cast = 0
    for p in model_student.t2u_model.parameters():
        if p.dtype == torch.float16:
            p.data = p.data.float()
            n_cast += 1
    print(f"  Cast {n_cast} trainable FP16 params to FP32")

    # Separate LR groups
    enc_params, dec_params, dur_params, scalar_params, head_params = [], [], [], [], []
    for name, p in model_student.t2u_model.named_parameters():
        if not p.requires_grad:
            continue
        if "duration_predictor" in name:
            dur_params.append(p)
        elif "pos_emb_alpha" in name:
            scalar_params.append(p)
        elif name == "lm_head.weight":
            head_params.append(p)
        elif name.startswith("model.decoder."):
            dec_params.append(p)
        else:
            enc_params.append(p)

    optimizer = torch.optim.AdamW(
        [
            {"params": enc_params,    "lr": base_lr,  "weight_decay": 0.01},
            {"params": dec_params,    "lr": base_lr,  "weight_decay": 0.01},
            {"params": dur_params,    "lr": dur_lr,   "weight_decay": 0.00},
            {"params": scalar_params, "lr": dur_lr,   "weight_decay": 0.00},
            {"params": head_params,   "lr": base_lr,  "weight_decay": 0.01},
        ],
        betas=(0.9, 0.98),
    )

    warmup_steps = max(1, int(0.10 * steps))
    def lr_lambda(step):
        if step < warmup_steps:
            return step / warmup_steps
        progress = (step - warmup_steps) / max(1, steps - warmup_steps)
        return 0.5 * (1.0 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)
    scaler    = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    model_student.train()
    optimizer.zero_grad(set_to_none=True)

    logs = []
    micro_step = 0
    opt_step   = 0
    running = {"nll": 0.0, "kd": 0.0, "len": 0.0, "loss": 0.0, "n": 0}

    total_micro = steps * grad_accum

    print(f"\nPhase 6C (CORRECTED) — {steps} optimizer steps, grad_accum={grad_accum}")
    print(f"  Primary loss: NLL against cached teacher unit IDs (.logits)")
    print(f"  KD supplement: {'enabled (0.20 weight)' if use_kd_supplement else 'disabled'}")
    print(f"  Length loss: normalized smooth-L1 (0.02 weight)\n")

    while micro_step < total_micro:
        sample, cache_entry = phase6_pick_training_pair(
            max_audio_sec=max_audio_sec, balanced=True
        )

        try:
            loss, metrics = phase6c_step_corrected(
                model_student=model_student,
                model_teacher=model_teacher if use_kd_supplement else None,
                sample=sample,
                cache_entry=cache_entry,
                student_device=student_device,
                teacher_device=teacher_device,
                autocast_dtype=autocast_dtype,
                use_kd_supplement=use_kd_supplement,
            )
        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                torch.cuda.empty_cache()
                print(f"  [OOM at micro {micro_step}] — skipping sample")
                micro_step += 1
                continue
            raise

        scaler.scale(loss / grad_accum).backward()
        micro_step += 1

        # Accumulate metrics
        for k, v in metrics.items():
            running[k] += v
        running["loss"] += loss.item()
        running["n"]    += 1

        if micro_step % grad_accum == 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model_student.t2u_model.parameters() if p.requires_grad],
                max_norm=1.0,
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)
            opt_step += 1

            if opt_step % log_every == 0:
                n = max(1, running["n"])
                avg = {k: running[k] / n for k in ["loss", "nll", "kd", "len"]}
                lr_now = scheduler.get_last_lr()[0]
                print(
                    f"  6c opt {opt_step:4d}/{steps} | "
                    f"loss={avg['loss']:.4f} | "
                    f"nll={avg['nll']:.4f} | "
                    f"kd={avg['kd']:.4f} | "
                    f"len={avg['len']:.4f} | "
                    f"lr={lr_now:.2e}"
                )
                logs.append({"step": opt_step, **avg, "lr": lr_now})
                running = {"nll": 0.0, "kd": 0.0, "len": 0.0, "loss": 0.0, "n": 0}

            if opt_step % eval_every == 0:
                model_student.eval()
                phase6_quick_eval(f"6c_step{opt_step:06d}", max_samples=16)
                model_student.train()

            if opt_step % save_every == 0:
                os.makedirs(ckpt_dir, exist_ok=True)
                ckpt_path = f"{ckpt_dir}/phase6_6c_step{opt_step:06d}.pt"
                torch.save({
                    "optimizer_step": opt_step,
                    "t2u_model": model_student.t2u_model.state_dict(),
                    "optimizer": optimizer.state_dict(),
                    "scheduler": scheduler.state_dict(),
                }, ckpt_path)
                print(f"  [saved] {ckpt_path}")

            if opt_step >= steps:
                break

    print(f"\nPhase 6C complete ({opt_step} optimizer steps).")
    return logs

def t2u_overlap_losses(student_out, teacher_out, temperature=2.0):
    """
    T2U distillation loss on overlapping positions.
    Uses last_hidden_state as unit logits (architecture-specific).
    """
    student_logits = student_out.last_hidden_state
    teacher_logits = teacher_out.last_hidden_state

    student_mask = student_out.padding_mask.bool()
    teacher_mask = teacher_out.padding_mask.bool()

    student_len = student_mask.sum(1).long()
    teacher_len = teacher_mask.sum(1).long()
    common_len  = torch.minimum(student_len, teacher_len)

    total_kl = student_logits.new_zeros(())
    total_ce = student_logits.new_zeros(())
    valid = 0

    for b in range(student_logits.size(0)):
        L = int(common_len[b].item())
        if L < 2:
            continue

        s = student_logits[b, :L]
        t = teacher_logits[b, :L]

        total_kl = total_kl + F.kl_div(
            F.log_softmax(s / temperature, dim=-1),
            F.softmax(t / temperature, dim=-1),
            reduction="batchmean",
        ) * (temperature ** 2)

        teacher_hard = t.argmax(dim=-1)
        total_ce = total_ce + F.cross_entropy(s, teacher_hard)
        valid += 1

    if valid == 0:
        raise RuntimeError("No valid T2U overlap found in batch.")

    total_kl = total_kl / valid
    total_ce = total_ce / valid

    # EXACTLY per planning doc — smooth L1 on absolute lengths
    total_len = F.smooth_l1_loss(student_len.float(), teacher_len.float())

    return total_kl, total_ce, total_len

def make_t2u_param_groups(model_student, base_lr=8e-5, dur_lr=1e-4, scalar_lr=1e-4, head_lr=8e-5):
    enc_params = []
    dec_params = []
    dur_params = []
    scalar_params = []
    head_params = []

    for name, p in model_student.t2u_model.named_parameters():
        if not p.requires_grad:
            continue
        if 'duration_predictor' in name:
            dur_params.append(p)
        elif 'pos_emb_alpha' in name:
            scalar_params.append(p)
        elif name == 'lm_head.weight':
            head_params.append(p)
        elif name.startswith('model.decoder.'):
            dec_params.append(p)
        else:
            enc_params.append(p)

    groups = []
    if enc_params:
        groups.append({'params': enc_params, 'lr': base_lr, 'weight_decay': 0.01})
    if dec_params:
        groups.append({'params': dec_params, 'lr': base_lr, 'weight_decay': 0.01})
    if dur_params:
        groups.append({'params': dur_params, 'lr': dur_lr, 'weight_decay': 0.00})
    if scalar_params:
        groups.append({'params': scalar_params, 'lr': scalar_lr, 'weight_decay': 0.00})
    if head_params:
        groups.append({'params': head_params, 'lr': head_lr, 'weight_decay': 0.01})
    return groups

def text_recovery_step(sample, cache_entry, use_teacher_text):
    """
    Text recovery training step.
    Cache entries are already remapped to 22K vocab.
    """
    audio_inputs = phase6_prepare_audio_inputs(sample, student_device)
    
    if use_teacher_text:
        labels = cache_entry['teacher_text_sequences'].unsqueeze(0).to(student_device)
        labels = labels.masked_fill(labels == _NEW_PAD_ID, -100)
    else:
        labels = build_target_labels(processor, [sample['ref']], sample['tgt_lang'], student_device)

    outputs = model_student(
        **audio_inputs,
        labels=labels,
        use_cache=False,
        output_attentions=False,
        output_hidden_states=False,
        return_dict=True,
    )
    return outputs.loss


def run_text_recovery_stage(
    stage_key,
    title,
    steps,
    max_audio_sec,
    text_lr,
    speech_lr=None,
    kd_prob=0.5,
    resume_from_step=0,
):
    freeze_all_student()
    enable_lora_params(model_student.text_decoder, 'text_decoder')
    if speech_lr is not None:
        enable_lora_params(model_student.speech_encoder, 'speech_encoder')

    optimizer_groups = [{'params': trainable_named_params(model_student.text_decoder),
                         'lr': text_lr, 'weight_decay': 0.01}]
    if speech_lr is not None:
        optimizer_groups.append({'params': trainable_named_params(model_student.speech_encoder),
                                 'lr': speech_lr, 'weight_decay': 0.01})

    optimizer = torch.optim.AdamW(optimizer_groups, betas=(0.9, 0.98))
    scheduler = make_cosine_scheduler(optimizer, steps)
    scaler    = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    # ── Derived cadences (clamped so eval/save never fire more often than log) ─
    log_every  = max(1,        LOG_EVERY)
    eval_every = max(log_every, EVAL_EVERY)
    save_every = max(log_every, SAVE_EVERY)

    # ── Resume ────────────────────────────────────────────────────────────────
    if resume_from_step > 0:
        ckpt = phase6_load_latest_local_checkpoint(f'phase6_{stage_key}')
        if ckpt is None:
            raise RuntimeError(
                f'resume_from_step={resume_from_step} requested but no checkpoint found '
                f'for phase6_{stage_key}'
            )
        model_student.text_decoder.load_state_dict(ckpt['text_decoder'])
        if speech_lr is not None and 'speech_encoder' in ckpt:
            model_student.speech_encoder.load_state_dict(ckpt['speech_encoder'])
        optimizer.load_state_dict(ckpt['optimizer'])
        if ckpt.get('logs'):
            phase6_logs[stage_key] = ckpt['logs']
        for _ in range(resume_from_step):
            scheduler.step()
        print(f'  Resumed {stage_key} from optimizer step {resume_from_step}/{steps}')
        # checking if LR scheduler is working
        print(f'  Scheduler base_lrs: {scheduler.base_lrs}')   # must show [1.5e-05, ...]
        print(f'  Current param_group lrs: {[g["lr"] for g in optimizer.param_groups]}')

    model_student.train()
    optimizer.zero_grad(set_to_none=True)

    print(f'\n[{stage_key}] {title}')
    print(f'  optimizer steps : {resume_from_step} → {steps}')
    print(f'  fwd passes left : {(steps - resume_from_step) * GRAD_ACCUM}')
    print(f'  log/eval/save   : every {log_every}/{eval_every}/{save_every} opt steps')
    print(f'  max_audio={max_audio_sec}s | trainable={count_trainable_params(model_student):.2f}M')

    start_micro = resume_from_step * GRAD_ACCUM
    total_micro = steps            * GRAD_ACCUM

    for micro_step in range(start_micro, total_micro):
        sample, cache_entry  = phase6_pick_training_pair(max_audio_sec=max_audio_sec, balanced=True)
        use_teacher_text     = random.random() < kd_prob

        try:
            with torch.cuda.amp.autocast(dtype=autocast_dtype):
                loss = text_recovery_step(sample, cache_entry, use_teacher_text=use_teacher_text)
            scaler.scale(loss / GRAD_ACCUM).backward()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                safe_gc()
                phase6_raise_oom(stage_key, micro_step + 1, max_audio_sec,
                                 extra='lower the stage audio cap')
            raise

        phase6_logs[stage_key].append({
            'micro_step':       micro_step + 1,
            'loss':             float(loss.detach().cpu()),
            'use_teacher_text': bool(use_teacher_text),
            'text_lr':          optimizer.param_groups[0]['lr'],
        })

        if (micro_step + 1) % GRAD_ACCUM == 0:
            opt_step = (micro_step + 1) // GRAD_ACCUM

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model_student.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            # ── Log ───────────────────────────────────────────────────────────
            if opt_step % log_every == 0:
                recent   = phase6_logs[stage_key][-(log_every * GRAD_ACCUM):]
                avg_loss = np.mean([r['loss']             for r in recent])
                kd_ratio = np.mean([r['use_teacher_text'] for r in recent])
                print(
                    f"  [{stage_key}] opt {opt_step:>4}/{steps} | "
                    f"loss={avg_loss:.4f} | KD={kd_ratio:.0%} | "
                    f"lr={optimizer.param_groups[0]['lr']:.2e}"
                )

            # ── Eval ──────────────────────────────────────────────────────────
            if opt_step % eval_every == 0:
                phase6_quick_eval(f'{stage_key}_step{opt_step}', max_samples=16)

            # ── Checkpoint ────────────────────────────────────────────────────
            if opt_step % save_every == 0:
                state = {
                    'stage':          stage_key,
                    'optimizer_step': opt_step,
                    'steps_total':    steps,
                    'logs':           phase6_logs[stage_key],
                    'text_decoder':   model_student.text_decoder.state_dict(),
                    'optimizer':      optimizer.state_dict(),
                }
                if speech_lr is not None:
                    state['speech_encoder'] = model_student.speech_encoder.state_dict()
                save_checkpoint(state, f'phase6_{stage_key}', opt_step)

    return phase6_logs[stage_key]


def ensure_teacher_loaded():
    global model_teacher
    if 'model_teacher' in globals() and model_teacher is not None:
        return model_teacher

    print('Reloading teacher model for Phase 6 KD...')
    try:
        model_teacher, _ = load_model_from_drive('phase1_vocab_5lang', device_map=None)
    except Exception:
        print('  Teacher checkpoint not found, loading HF base teacher.')
        model_teacher, _ = load_base_model()
    model_teacher = move_model_to_device(model_teacher, teacher_device)
    model_teacher.eval()
    for p in model_teacher.parameters():
        p.requires_grad_(False)
    disable_generation_cache(model_teacher)
    print(f'  Teacher device: {next(model_teacher.parameters()).device}')
    return model_teacher

def ensure_trainable_fp32():
    """Cast all trainable parameters to FP32 to avoid GradScaler FP16 grad errors."""
    count = 0
    for p in model_student.parameters():
        if p.requires_grad and p.dtype == torch.float16:
            p.data = p.data.float()
            count += 1
    print(f'  Cast {count} trainable FP16 params to FP32')

def run_t2u_recovery_stage(
    stage_key,
    title,
    steps,
    max_audio_sec,
    resume_from_step=0,
):
    ensure_teacher_loaded()

    if PHASE6_T2U_TRAIN_MODE == "selective":
        mark_t2u_selective_trainable()
    else:
        mark_t2u_trainable_full()

    ensure_trainable_fp32()

    # STRICT per planning doc: 8e-5 base, 1e-4 duration/scalar
    optimizer = torch.optim.AdamW(
        make_t2u_param_groups(model_student),  # uses defaults: base_lr=8e-5, dur_lr=1e-4, scalar_lr=1e-4, head_lr=8e-5
        betas=(0.9, 0.98),
    )
    scheduler = make_cosine_scheduler(optimizer, steps)
    scaler    = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    log_every  = max(1,        LOG_EVERY)
    eval_every = max(log_every, EVAL_EVERY)
    save_every = max(log_every, SAVE_EVERY)

    # ── Resume ────────────────────────────────────────────────────────────────
    if resume_from_step > 0:
        ckpt = phase6_load_latest_local_checkpoint(f"phase6_{stage_key}")
        if ckpt is None:
            raise RuntimeError(
                f"resume_from_step={resume_from_step} requested but no checkpoint found "
                f"for phase6_{stage_key}"
            )
        model_student.t2u_model.load_state_dict(ckpt["t2u_model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        if ckpt.get("logs"):
            phase6_logs[stage_key] = ckpt["logs"]
        for _ in range(resume_from_step):
            scheduler.step()
        print(f"  Resumed {stage_key} from optimizer step {resume_from_step}/{steps}")

    model_student.train()
    optimizer.zero_grad(set_to_none=True)

    print(f"\n[{stage_key}] {title}")
    print(f"  optimizer steps : {resume_from_step} → {steps}")
    print(f"  fwd passes left : {(steps - resume_from_step) * GRAD_ACCUM}")
    print(f"  log/eval/save   : every {log_every}/{eval_every}/{save_every} opt steps")
    print(f"  max_audio={max_audio_sec}s | trainable={count_trainable_params(model_student):.2f}M")

    start_micro = resume_from_step * GRAD_ACCUM
    total_micro = steps            * GRAD_ACCUM

    for micro_step in range(start_micro, total_micro):
        sample, cache_entry    = phase6_pick_training_pair(max_audio_sec=max_audio_sec, balanced=True)
        teacher_text_sequences = cache_entry["teacher_text_sequences"].unsqueeze(0)
        audio_inputs_student   = phase6_prepare_audio_inputs(sample, student_device)
        audio_inputs_teacher   = {k: v.to(teacher_device) for k, v in audio_inputs_student.items()}

        try:
            # ── Teacher path (GPU1, always no_grad) ───────────────────────────
            with torch.no_grad():
                teacher_cond = build_t2u_conditioning_from_sequences(
                    model_teacher,
                    input_features=audio_inputs_teacher["input_features"],
                    attention_mask=audio_inputs_teacher.get("attention_mask"),
                    text_sequences=teacher_text_sequences.to(teacher_device),
                )
                with torch.cuda.amp.autocast(dtype=autocast_dtype):
                    teacher_t2u = model_teacher.t2u_model(
                        inputs_embeds=teacher_cond["t2u_input_embeds"],
                        attention_mask=teacher_cond["t2u_attention_mask"],
                        char_input_ids=teacher_cond["t2u_char_input_ids"],
                        char_count_per_id=teacher_cond["t2u_char_count_per_id"],
                        output_attentions=False,
                        output_hidden_states=False,
                        return_dict=True,
                    )

            # ── Student conditioning (GPU0) — FROZEN path, NO_GRAD ─────────────
            # STRICT per planning doc: use model_student's own text_decoder
            with torch.no_grad():
                student_cond = build_t2u_conditioning_from_sequences(
                    model_student,
                    input_features=audio_inputs_student["input_features"],
                    attention_mask=audio_inputs_student.get("attention_mask"),
                    text_sequences=teacher_text_sequences.to(student_device),
                )
            t2u_inputs_embeds = student_cond["t2u_input_embeds"].detach()

            # ── Student T2U (GPU0, trainable) ─────────────────────────────────
            with torch.cuda.amp.autocast(dtype=autocast_dtype):
                student_t2u = model_student.t2u_model(
                    inputs_embeds=t2u_inputs_embeds,
                    attention_mask=student_cond["t2u_attention_mask"],
                    char_input_ids=student_cond["t2u_char_input_ids"],
                    char_count_per_id=student_cond["t2u_char_count_per_id"],
                    output_attentions=False,
                    output_hidden_states=False,
                    return_dict=True,
                )

                teacher_t2u.last_hidden_state = (
                    teacher_t2u.last_hidden_state.to(student_device)
                    if hasattr(teacher_t2u, "last_hidden_state") and teacher_t2u.last_hidden_state is not None
                    else None
                )
                teacher_t2u.padding_mask = teacher_t2u.padding_mask.to(student_device)

                t2u_soft, t2u_hard, t2u_len = t2u_overlap_losses(student_t2u, teacher_t2u)

                # EXACTLY per planning doc
                loss = 0.60 * t2u_soft + 0.30 * t2u_hard + 0.10 * t2u_len

            scaler.scale(loss / GRAD_ACCUM).backward()

        except RuntimeError as e:
            if "out of memory" in str(e).lower():
                safe_gc()
                phase6_raise_oom(stage_key, micro_step + 1, max_audio_sec,
                                 extra='set PHASE6_T2U_TRAIN_MODE="selective"')
            raise

        phase6_logs[stage_key].append({
            "micro_step": micro_step + 1,
            "loss":       float(loss.detach().cpu()),
            "t2u_soft":   float(t2u_soft.detach().cpu()),
            "t2u_hard":   float(t2u_hard.detach().cpu()),
            "t2u_len":    float(t2u_len.detach().cpu()),
            "lr":         optimizer.param_groups[0]["lr"],
        })

        if (micro_step + 1) % GRAD_ACCUM == 0:
            opt_step = (micro_step + 1) // GRAD_ACCUM

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model_student.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            if opt_step % log_every == 0:
                recent   = phase6_logs[stage_key][-(log_every * GRAD_ACCUM):]
                avg_loss = np.mean([r["loss"]     for r in recent])
                avg_soft = np.mean([r["t2u_soft"] for r in recent])
                avg_hard = np.mean([r["t2u_hard"] for r in recent])
                avg_len  = np.mean([r["t2u_len"]  for r in recent])
                print(
                    f"  [{stage_key}] opt {opt_step:>4}/{steps} | "
                    f"loss={avg_loss:.4f} | soft={avg_soft:.4f} | "
                    f"hard={avg_hard:.4f} | len={avg_len:.4f} | "
                    f"lr={optimizer.param_groups[0]['lr']:.2e}"
                )

            if opt_step % eval_every == 0:
                phase6_quick_eval(f"{stage_key}_step{opt_step}", max_samples=16)

            if opt_step % save_every == 0:
                save_checkpoint(
                    {
                        "stage":          stage_key,
                        "optimizer_step": opt_step,
                        "steps_total":    steps,
                        "mode":           PHASE6_T2U_TRAIN_MODE,
                        "logs":           phase6_logs[stage_key],
                        "t2u_model":      model_student.t2u_model.state_dict(),
                        "optimizer":      optimizer.state_dict(),
                    },
                    f"phase6_{stage_key}", opt_step,
                )

    return phase6_logs[stage_key]


def run_joint_polish_stage(
    stage_key,
    title,
    steps,
    max_audio_sec,
    resume_from_step=0,
):
    ensure_teacher_loaded()
    freeze_all_student()
    enable_lora_params(model_student.text_decoder, 'text_decoder')
    if PHASE6_T2U_TRAIN_MODE == 'selective':
        mark_t2u_selective_trainable()
    else:
        mark_t2u_trainable_full()

    ensure_trainable_fp32()  # <-- ADD THIS
    
    enable_lora_params(model_student.text_decoder, 'text_decoder')

    text_params = trainable_named_params(model_student.text_decoder)
    t2u_groups  = make_t2u_param_groups(model_student, base_lr=4e-5, dur_lr=5e-5,
                                         scalar_lr=5e-5, head_lr=4e-5)
    optimizer   = torch.optim.AdamW(
        [{'params': text_params, 'lr': 1e-5, 'weight_decay': 0.01}] + t2u_groups,
        betas=(0.9, 0.98),
    )
    scheduler = make_cosine_scheduler(optimizer, steps)
    scaler    = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

    log_every  = max(1,        LOG_EVERY)
    eval_every = max(log_every, EVAL_EVERY)
    save_every = max(log_every, SAVE_EVERY)

    # ── Resume ────────────────────────────────────────────────────────────────
    if resume_from_step > 0:
        ckpt = phase6_load_latest_local_checkpoint(f'phase6_{stage_key}')
        if ckpt is None:
            raise RuntimeError(
                f'resume_from_step={resume_from_step} requested but no checkpoint found '
                f'for phase6_{stage_key}'
            )
        model_student.text_decoder.load_state_dict(ckpt['text_decoder'])
        model_student.t2u_model.load_state_dict(ckpt['t2u_model'])
        optimizer.load_state_dict(ckpt['optimizer'])
        if ckpt.get('logs'):
            phase6_logs[stage_key] = ckpt['logs']
        for _ in range(resume_from_step):
            scheduler.step()
        print(f'  Resumed {stage_key} from optimizer step {resume_from_step}/{steps}')

    model_student.train()
    optimizer.zero_grad(set_to_none=True)

    print(f'\n[{stage_key}] {title}')
    print(f'  optimizer steps : {resume_from_step} → {steps}')
    print(f'  fwd passes left : {(steps - resume_from_step) * GRAD_ACCUM}')
    print(f'  log/eval/save   : every {log_every}/{eval_every}/{save_every} opt steps')
    print(f'  max_audio={max_audio_sec}s | trainable={count_trainable_params(model_student):.2f}M')

    start_micro = resume_from_step * GRAD_ACCUM
    total_micro = steps            * GRAD_ACCUM

    for micro_step in range(start_micro, total_micro):
        sample, cache_entry    = phase6_pick_training_pair(max_audio_sec=max_audio_sec, balanced=True)
        teacher_text_sequences = cache_entry['teacher_text_sequences'].unsqueeze(0)
        use_teacher_text       = random.random() < PHASE6_TEXT_KD_PROB
        target_text            = cache_entry['teacher_text_str'] if use_teacher_text else sample['ref']

        audio_inputs_student = phase6_prepare_audio_inputs(sample, student_device)
        audio_inputs_teacher = {k: v.to(teacher_device) for k, v in audio_inputs_student.items()}
        labels               = build_target_labels(processor, [target_text], sample['tgt_lang'],
                                                   student_device)

        try:
            with torch.no_grad():
                teacher_cond = build_t2u_conditioning_from_sequences(
                    model_teacher,
                    input_features=audio_inputs_teacher['input_features'],
                    attention_mask=audio_inputs_teacher.get('attention_mask'),
                    text_sequences=teacher_text_sequences.to(teacher_device),
                )
                with torch.cuda.amp.autocast(dtype=autocast_dtype):
                    teacher_t2u = model_teacher.t2u_model(
                        inputs_embeds=teacher_cond['t2u_input_embeds'],
                        attention_mask=teacher_cond['t2u_attention_mask'],
                        char_input_ids=teacher_cond['t2u_char_input_ids'],
                        char_count_per_id=teacher_cond['t2u_char_count_per_id'],
                        output_attentions=False, output_hidden_states=False, return_dict=True,
                    )

            with torch.cuda.amp.autocast(dtype=autocast_dtype):
                text_outputs = model_student(
                    **audio_inputs_student,
                    labels=labels,
                    use_cache=False,
                    output_attentions=False, output_hidden_states=False, return_dict=True,
                )
                student_cond = build_t2u_conditioning_from_sequences(
                    model_student,
                    input_features=audio_inputs_student['input_features'],
                    attention_mask=audio_inputs_student.get('attention_mask'),
                    text_sequences=teacher_text_sequences.to(student_device),
                )
                student_t2u = model_student.t2u_model(
                    inputs_embeds=student_cond['t2u_input_embeds'],
                    attention_mask=student_cond['t2u_attention_mask'],
                    char_input_ids=student_cond['t2u_char_input_ids'],
                    char_count_per_id=student_cond['t2u_char_count_per_id'],
                    output_attentions=False, output_hidden_states=False, return_dict=True,
                )
                teacher_t2u.last_hidden_state = teacher_t2u.last_hidden_state.to(student_device)
                teacher_t2u.padding_mask      = teacher_t2u.padding_mask.to(student_device)
                t2u_soft, t2u_hard, t2u_len   = t2u_overlap_losses(student_t2u, teacher_t2u)
                text_loss = text_outputs.loss
                loss = 0.35 * text_loss + 0.40 * t2u_soft + 0.20 * t2u_hard + 0.05 * t2u_len
            scaler.scale(loss / GRAD_ACCUM).backward()
        except RuntimeError as e:
            if 'out of memory' in str(e).lower():
                safe_gc()
                phase6_raise_oom(stage_key, micro_step + 1, max_audio_sec,
                                 extra='reduce Stage 6D audio cap')
            raise

        phase6_logs[stage_key].append({
            'micro_step':       micro_step + 1,
            'loss':             float(loss.detach().cpu()),
            'text_loss':        float(text_loss.detach().cpu()),
            't2u_soft':         float(t2u_soft.detach().cpu()),
            't2u_hard':         float(t2u_hard.detach().cpu()),
            't2u_len':          float(t2u_len.detach().cpu()),
            'use_teacher_text': bool(use_teacher_text),
            'lr':               optimizer.param_groups[0]['lr'],
        })

        if (micro_step + 1) % GRAD_ACCUM == 0:
            opt_step = (micro_step + 1) // GRAD_ACCUM

            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                [p for p in model_student.parameters() if p.requires_grad],
                MAX_GRAD_NORM,
            )
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            scheduler.step()

            # ── Log ───────────────────────────────────────────────────────────
            if opt_step % log_every == 0:
                recent   = phase6_logs[stage_key][-(log_every * GRAD_ACCUM):]
                avg_loss = np.mean([r['loss']      for r in recent])
                avg_text = np.mean([r['text_loss'] for r in recent])
                avg_soft = np.mean([r['t2u_soft']  for r in recent])
                avg_hard = np.mean([r['t2u_hard']  for r in recent])
                print(
                    f"  [{stage_key}] opt {opt_step:>4}/{steps} | "
                    f"loss={avg_loss:.4f} | text={avg_text:.4f} | "
                    f"soft={avg_soft:.4f} | hard={avg_hard:.4f} | "
                    f"lr={optimizer.param_groups[0]['lr']:.2e}"
                )

            # ── Eval ──────────────────────────────────────────────────────────
            if opt_step % eval_every == 0:
                phase6_quick_eval(f'{stage_key}_step{opt_step}', max_samples=16)

            # ── Checkpoint ────────────────────────────────────────────────────
            if opt_step % save_every == 0:
                save_checkpoint(
                    {
                        'stage':          stage_key,
                        'optimizer_step': opt_step,
                        'steps_total':    steps,
                        'mode':           PHASE6_T2U_TRAIN_MODE,
                        'logs':           phase6_logs[stage_key],
                        'text_decoder':   model_student.text_decoder.state_dict(),
                        't2u_model':      model_student.t2u_model.state_dict(),
                        'optimizer':      optimizer.state_dict(),
                    },
                    f'phase6_{stage_key}', opt_step,
                )

    return phase6_logs[stage_key]


print('Loading Phase 5 student model...')
model_student, processor = load_model_from_drive('phase5_dec_14L', device_map=None)
disable_generation_cache(model_student)

print('Loading teacher model (vocab pruned) for Phase 6A cache build...')
try:
    model_teacher, _ = load_model_from_drive('phase1_vocab_5lang', device_map=None)
except Exception:
    print('  Teacher checkpoint not found, NOT loading HF base teacher.')
    # model_teacher, _ = load_base_model()
model_teacher = move_model_to_device(model_teacher, teacher_device)
model_teacher.eval()
for p in model_teacher.parameters():
    p.requires_grad_(False)
disable_generation_cache(model_teacher)

speech_lora_cfg = make_lora_config(
    build_speech_encoder_lora_targets(model_student),
    r=16,
    alpha=32,
)
text_lora_cfg = make_lora_config(
    build_text_decoder_lora_targets(model_student),
    r=32,
    alpha=64,
)

model_student.speech_encoder = wrap_with_lora_if_needed(
    model_student.speech_encoder,
    speech_lora_cfg,
    'speech_encoder',
)
model_student.text_decoder = wrap_with_lora_if_needed(
    model_student.text_decoder,
    text_lora_cfg,
    'text_decoder',
)
freeze_all_student()
model_student = move_model_to_device(model_student, student_device)

# Build once at startup, reuse everywhere
_old_to_new = {
    int(old_id): new_id
    for new_id, old_id in enumerate(model_student._vocab_remap_to_old.tolist())
}
_student_vocab_size = model_student.text_decoder.get_base_model().embed_tokens.num_embeddings
print(f"Remap table built: {len(_old_to_new)} entries, student vocab={_student_vocab_size}")


def build_target_labels(processor, text_list, tgt_lang, device):
    tok = processor.tokenizer(
        text_target=text_list,
        tgt_lang=tgt_lang,
        return_tensors='pt',
        padding=True,
    )
    labels = tok['input_ids'].clone()  # still 256K IDs

    # Remap every ID through old->new map
    remapped = torch.full_like(labels, -100)  # default: ignore
    for i in range(labels.shape[0]):
        for j in range(labels.shape[1]):
            old_id = int(labels[i, j].item())
            if old_id == processor.tokenizer.pad_token_id:
                remapped[i, j] = -100
            elif old_id in _old_to_new:
                remapped[i, j] = _old_to_new[old_id]
            # else: pruned token, stays -100 (ignored in loss)

    return remapped.to(device)

# ═══════════════════════════════════════════════════════════════════════════════
# GLOBAL REMAP CONSTANTS — run once after model loading
# ═══════════════════════════════════════════════════════════════════════════════

# Precompute new→old dense tensor for processor.decode ops
_NEW_TO_OLD_TENSOR = model_student._vocab_remap_to_old  # shape [22767]

# Special token IDs in NEW vocab (for masking without touching processor)
_NEW_PAD_ID = _old_to_new[processor.tokenizer.pad_token_id]   # 0
_NEW_EOS_ID = _old_to_new[processor.tokenizer.eos_token_id]   # 3

print(f"Global remap constants | _NEW_PAD_ID={_NEW_PAD_ID} | _NEW_EOS_ID={_NEW_EOS_ID}")

print(f'Student device: {next(model_student.parameters()).device}')
print(f'Teacher device: {next(model_teacher.parameters()).device}')
print_model_breakdown(model_student, 'Phase 6 student with LoRA wrappers')
gpu_mem()


Loading Phase 5 student model...
[model] Not in local cache — pulling from remote...
[rclone] Pulled phase5_dec_14L → /kaggle/working/models/phase5_dec_14L
[model] Loading phase5_dec_14L from /kaggle/working/models/phase5_dec_14L ...


Loading weights:   0%|          | 0/1234 [00:00<?, ?it/s]

  Restored custom state: ['_vocab_remap_to_old']
[model] Loaded phase5_dec_14L.
Loading teacher model (vocab pruned) for Phase 6A cache build...
[model] Not in local cache — pulling from remote...
[rclone] Pulled phase1_vocab_5lang → /kaggle/working/models/phase1_vocab_5lang
[model] Loading phase1_vocab_5lang from /kaggle/working/models/phase1_vocab_5lang ...


Loading weights:   0%|          | 0/1846 [00:00<?, ?it/s]

  Restored custom state: ['_vocab_remap_to_old']
[model] Loaded phase1_vocab_5lang.
speech_encoder LoRA attached.
trainable params: 6,605,312 || all params: 399,849,152 || trainable%: 1.6520
text_decoder LoRA attached.
trainable params: 15,597,568 || all params: 391,564,288 || trainable%: 3.9834
Remap table built: 22767 entries, student vocab=22767
Global remap constants | _NEW_PAD_ID=0 | _NEW_EOS_ID=3
Student device: cuda:0
Teacher device: cuda:1

--- Phase 6 student with LoRA wrappers ---
  speech_encoder                         399.8M  ( 38.0%)
  text_decoder                           391.6M  ( 37.2%)
  t2u_model                              219.8M  ( 20.9%)
  vocoder                                 41.9M  (  4.0%)
  shared                                  23.3M  (  2.2%)
  lm_head                                 23.3M  (  2.2%)
  TOTAL                                 1053.1M
---
  GPU0: 2.18GB alloc / 2.18GB reserved
  GPU1: 3.16GB alloc / 3.17GB reserved


In [31]:
def phase6_cache_checkpoint_name(split_name):
    return f'{PHASE6_CACHE_PREFIX}_{split_name}'


def phase6_cache_manifest_name(split_name):
    return f'{PHASE6_CACHE_PREFIX}_{split_name}_manifest'


def phase6_save_checkpoint_local(state, name, step=0, keep=3):
    fname = f'{name}_step{step:06d}.pt'
    path = f'{CKPT_DIR}/{fname}'
    torch.save(state, path)
    mb = os.path.getsize(path) / 1e6
    print(f'[ckpt-local] Saved {fname} ({mb:.1f} MB)')
    old = sorted(glob.glob(f'{CKPT_DIR}/{name}_step*.pt'))
    for f in old[:-keep]:
        if os.path.exists(f):
            os.remove(f)
    return path


def phase6_rclone_copy_checkpoint_family(prefixes, direction='push'):
    if not ON_KAGGLE:
        return

    if isinstance(prefixes, str):
        prefixes = [prefixes]

    include_args = ' '.join([f'--include "{prefix}_step*.pt"' for prefix in prefixes])
    if direction == 'push':
        src = f'{CKPT_DIR}/'
        dst = f'{GDRIVE_ROOT}/checkpoints/'
        verb = 'push'
    else:
        src = f'{GDRIVE_ROOT}/checkpoints/'
        dst = f'{CKPT_DIR}/'
        verb = 'pull'

    cmd = (
        f'rclone copy "{src}" "{dst}" {include_args} '
        f'--transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M'
    )
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'[rclone] phase6 cache {verb} failed: {result.stderr[:300]}')
    print(f'[rclone] Phase 6 cache {verb} OK for: {prefixes}')


def phase6_load_latest_local_checkpoint(name):
    files = sorted(glob.glob(f'{CKPT_DIR}/{name}_step*.pt'))
    if not files:
        return None
    return torch.load(files[-1], map_location='cpu', weights_only=False)


def phase6_load_manifest(split_name):
    name = phase6_cache_manifest_name(split_name)
    manifest = phase6_load_latest_local_checkpoint(name)
    if manifest is None and ON_KAGGLE:
        print(f'No local manifest for {split_name}. Pulling Phase 6 cache family from Drive...')
        phase6_rclone_copy_checkpoint_family(
            [phase6_cache_checkpoint_name(split_name), phase6_cache_manifest_name(split_name)],
            direction='pull',
        )
        manifest = phase6_load_latest_local_checkpoint(name)

    if manifest is not None:
        return manifest

    return {
        'split': split_name,
        'total_cached': 0,
        'cache_index': {},
        'pair_to_keys': {},
        'num_shards': 0,
        'synced_until': 0,
    }


def phase6_save_manifest_local(split_name, manifest):
    return phase6_save_checkpoint_local(
        manifest,
        phase6_cache_manifest_name(split_name),
        step=manifest['num_shards'],
        keep=8,
    )


def phase6_save_cache_shard_local(split_name, shard_idx, entries, total_cached):
    return phase6_save_checkpoint_local(
        {
            'split': split_name,
            'shard_idx': shard_idx,
            'entries': entries,
            'total_cached': total_cached,
        },
        phase6_cache_checkpoint_name(split_name),
        step=shard_idx,
        keep=100000,
    )


def build_teacher_cache_entry(model_teacher, sample):
    teacher_inputs = phase6_prepare_audio_inputs(sample, teacher_device)
    out = teacher_generate_tokens(model_teacher, teacher_inputs, tgt_lang=sample['tgt_lang'])

    if out.unit_sequences is None:
        return None, 'teacher_returned_no_unit_sequence'

    teacher_text_sequences = out.sequences[0].detach().cpu()  # NEW IDs
    
    if teacher_text_sequences.numel() == 0:
        return None, 'empty_teacher_sequence'
    
    if teacher_text_sequences.numel() > 512:
        return None, f'teacher_sequence_too_long:{teacher_text_sequences.numel()}'

    teacher_unit_sequences = out.unit_sequences[0].detach().cpu()
    
    # ── FIX: processor.batch_decode needs OLD IDs ───────────────────────────
    old_ids = _NEW_TO_OLD_TENSOR[teacher_text_sequences]
    teacher_text_str = processor.batch_decode(
        old_ids.unsqueeze(0),
        skip_special_tokens=True,
    )[0].strip()
    # ─────────────────────────────────────────────────────────────────────────

    unit_len = int(teacher_unit_sequences.numel())

    if not teacher_text_str:
        return None, 'empty_teacher_text'
    if unit_len < PHASE6_MIN_TEACHER_UNIT_TOKENS:
        return None, f'unit_sequence_too_short:{unit_len}'
    if unit_len > PHASE6_MAX_TEACHER_UNIT_TOKENS:
        return None, f'unit_sequence_too_long:{unit_len}'

    return {
        'sample_key': phase6_sample_key(sample),
        'sample_id': sample['id'],
        'src_lang': sample['src_lang'],
        'tgt_lang': sample['tgt_lang'],
        'teacher_text_sequences': teacher_text_sequences,  # keep NEW IDs
        'teacher_text_str': teacher_text_str,
        'teacher_unit_sequences': teacher_unit_sequences,
        'audio_len_s': len(sample['wav']) / 16000.0,
    }, None


def build_or_load_phase6_cache(split_name, samples, shard_size=PHASE6_CACHE_SHARD_SIZE):
    manifest = phase6_load_manifest(split_name)
    cache_index = manifest.get('cache_index', {})
    pair_to_keys = defaultdict(list)
    for pair, keys in manifest.get('pair_to_keys', {}).items():
        pair_to_keys[pair].extend(keys)
    skipped_count = manifest.get('skipped_count', 0)
    total_samples = len(samples)
    sync_every = max(1, math.ceil(total_samples / PHASE6_CACHE_SYNC_PARTS))
    next_sync_target = max(sync_every, ((len(cache_index) // sync_every) + 1) * sync_every)
    next_sync_target = min(total_samples, next_sync_target)
    print(
        f'Existing {split_name} cache index: {len(cache_index)} samples | '
        f'shard_size={shard_size} | sync_every~{sync_every} samples | skipped={skipped_count}'
    )
    # If cache marked complete, skip entirely
    if manifest.get('cache_complete', False):
        print(f'✅ Cache already marked complete. Cached={len(cache_index)}')
        return manifest
    # If we already have all cacheable samples (cached + known skipped >= total), treat as done
    if len(cache_index) + skipped_count >= total_samples and len(cache_index) > 0:
        print(f'✅ Cache appears complete (cached {len(cache_index)} + skipped {skipped_count} >= {total_samples}). Done.')
        manifest['cache_complete'] = True
        phase6_save_manifest_local(split_name, manifest)
        return manifest
    buffer = []
    shard_idx = int(manifest.get('num_shards', 0))

    
    for idx in range(len(samples)):
        sample = samples[idx]
        sample_key = phase6_sample_key(sample)
        if sample_key in cache_index:
            print(f"already cached> {sample_key}")
            continue
        entry, err = build_teacher_cache_entry(model_teacher, sample)
        if entry is None:
            skipped_count += 1
            print(f'  skipped {skipped_count} samples (latest: {err}) | {sample_key}')
            continue
        offset = len(buffer)
        buffer.append(entry)
        pair = f"{entry['src_lang']}->{entry['tgt_lang']}"
        cache_index[sample_key] = {
            'split': split_name,
            'shard_idx': shard_idx,
            'offset': offset,
            'audio_len_s': entry['audio_len_s'],
            'src_lang': entry['src_lang'],
            'tgt_lang': entry['tgt_lang'],
        }
        pair_to_keys[pair].append(sample_key)
        if len(buffer) >= shard_size:
            phase6_save_cache_shard_local(
                split_name=split_name,
                shard_idx=shard_idx,
                entries=buffer,
                total_cached=len(cache_index),
            )
            shard_idx += 1
            manifest = {
                'split': split_name,
                'total_cached': len(cache_index),
                'cache_index': cache_index,
                'pair_to_keys': dict(pair_to_keys),
                'num_shards': shard_idx,
                'synced_until': manifest.get('synced_until', 0),
                'skipped_count': skipped_count,
            }
            phase6_save_manifest_local(split_name, manifest)
            buffer = []
            safe_gc()
        if len(cache_index) >= next_sync_target:
            manifest = {
                'split': split_name,
                'total_cached': len(cache_index),
                'cache_index': cache_index,
                'pair_to_keys': dict(pair_to_keys),
                'num_shards': shard_idx,
                'synced_until': len(cache_index),
                'skipped_count': skipped_count,
            }
            phase6_save_manifest_local(split_name, manifest)
            phase6_rclone_copy_checkpoint_family(
                [phase6_cache_checkpoint_name(split_name), phase6_cache_manifest_name(split_name)],
                direction='push',
            )
            print(f'  synced Phase 6A cache to Drive at {len(cache_index)}/{total_samples} samples')
            next_sync_target = min(total_samples, next_sync_target + sync_every)
        if (idx + 1) % 200 == 0:
            print(f'  cached {len(cache_index)}/{total_samples} {split_name} samples')
    if buffer:
        phase6_save_cache_shard_local(
            split_name=split_name,
            shard_idx=shard_idx,
            entries=buffer,
            total_cached=len(cache_index),
        )
        shard_idx += 1
    manifest = {
        'split': split_name,
        'total_cached': len(cache_index),
        'cache_index': cache_index,
        'pair_to_keys': dict(pair_to_keys),
        'num_shards': shard_idx,
        'synced_until': len(cache_index),
        'skipped_count': skipped_count,
        'cache_complete': True,
    }
    phase6_save_manifest_local(split_name, manifest)
    phase6_rclone_copy_checkpoint_family(
        [phase6_cache_checkpoint_name(split_name), phase6_cache_manifest_name(split_name)],
        direction='push',
    )
    print(f'✅ Done. Cached={len(cache_index)} | Skipped={skipped_count}')
    return manifest


print('Building or loading Phase 6A teacher cache...')
phase6_cache_manifest = build_or_load_phase6_cache('train', ft_samples, shard_size=PHASE6_CACHE_SHARD_SIZE)
phase6_cache_index = phase6_cache_manifest['cache_index']
phase6_cache_keys_by_pair = defaultdict(list)
for pair, keys in phase6_cache_manifest['pair_to_keys'].items():
    phase6_cache_keys_by_pair[pair].extend(keys)

phase6_sample_lookup = build_sample_lookup(ft_samples)
phase6_shard_cache = OrderedDict()

print(f"Phase 6A cache ready: {phase6_cache_manifest['total_cached']} entries")
print(f"  shards: {phase6_cache_manifest['num_shards']} | shard_size: {PHASE6_CACHE_SHARD_SIZE}")
print(f"  sync parts: {PHASE6_CACHE_SYNC_PARTS} | shard LRU limit: {PHASE6_SHARD_LRU_LIMIT}")
for pair, keys in sorted(phase6_cache_keys_by_pair.items()):
    print(f'  {pair:<12} {len(keys):>5} samples')

Building or loading Phase 6A teacher cache...
Existing train cache index: 9596 samples | shard_size=2048 | sync_every~2400 samples | skipped=4
✅ Cache already marked complete. Cached=9596
Phase 6A cache ready: 9596 entries
  shards: 5 | shard_size: 2048
  sync parts: 4 | shard LRU limit: 16
  arb->eng      1200 samples
  ben->eng      1200 samples
  cmn->eng      1200 samples
  eng->arb      1200 samples
  eng->ben      1200 samples
  eng->cmn      1196 samples
  eng->hin      1200 samples
  hin->eng      1200 samples


In [32]:
# print('Unloading teacher after cache build to free GPU1...')
# del model_teacher
# model_teacher = None
safe_gc()
gpu_mem()

  GPU0: 2.18GB alloc / 2.18GB reserved
  GPU1: 3.16GB alloc / 3.17GB reserved


In [33]:
# # Push remapped cache back to Drive
#     if ON_KAGGLE:
#         print("\nPushing remapped cache to Google Drive...")
#         phase6_rclone_copy_checkpoint_family(
#             [phase6_cache_checkpoint_name('train')],
#             direction='push',
#         )
#         print("✓ Pushed to Drive")

In [34]:
!ls checkpoints

all_detailed_summaries_step000000.pt
all_summaries_step000000.pt
phase0_benchmark_step000000.pt
phase1_benchmark_step000000.pt
phase2_benchmark_step000000.pt
phase2_enc_pruning_step000000.pt
phase3_benchmark_step000000.pt
phase3_laco_done_step000000.pt
phase4_benchmark_step000000.pt
phase4_enc_pruning_step000000.pt
phase5_benchmark_step000000.pt
phase5_dec_pruning_step000000.pt
phase6_6b1_step000200.pt
phase6_6b2_step001350.pt
phase6_teacher_cache_train_manifest_step000001.pt
phase6_teacher_cache_train_manifest_step000002.pt
phase6_teacher_cache_train_manifest_step000003.pt
phase6_teacher_cache_train_manifest_step000004.pt
phase6_teacher_cache_train_manifest_step000005.pt
phase6_teacher_cache_train_step000000.pt
phase6_teacher_cache_train_step000001.pt
phase6_teacher_cache_train_step000002.pt
phase6_teacher_cache_train_step000003.pt
phase6_teacher_cache_train_step000004.pt


In [35]:
# def verify_build_target_labels():
#     test_cases = [
#         ("hello world",       "eng"),
#         ("আমি ভালো আছি",       "ben"),
#         ("यह एक परीक्षण है",    "hin"),
#         ("هذا اختبار",          "arb"),
#         ("这是一个测试",           "cmn"),
#     ]

#     vocab_size = _student_vocab_size
#     print("=" * 60)
#     print("VERIFYING build_target_labels")
#     print(f"Student vocab size: {vocab_size}")
#     print("=" * 60)

#     all_ok = True
#     for text, lang in test_cases:
#         labels = build_target_labels(processor, [text], lang, torch.device('cpu'))
#         ids = labels[0].tolist()

#         # IDs that are not -100
#         active_ids = [i for i in ids if i != -100]
#         oob = [i for i in active_ids if i >= vocab_size or i < 0]
#         ignored = ids.count(-100)

#         status = "✓" if not oob else "✗ OOB"
#         print(f"\n  '{text}' ({lang})")
#         print(f"    raw ids     : {ids}")
#         print(f"    active ids  : {active_ids}  max={max(active_ids) if active_ids else 'n/a'}")
#         print(f"    ignored(-100): {ignored}/{len(ids)}")
#         print(f"    oob ids     : {oob if oob else 'none'}  {status}")

#         if oob:
#             all_ok = False

#         # Round-trip: decode active IDs back to text
#         if active_ids:
#             decoded = processor.tokenizer.decode(
#                 [_old_to_new_inv.get(i, i) for i in active_ids],  # back to old IDs for decode
#                 skip_special_tokens=True,
#             )
#             print(f"    decoded     : '{decoded}'")

#     # Also run one real training step in dry-run mode
#     print("\n" + "=" * 60)
#     print("DRY-RUN TRAINING STEP")
#     print("=" * 60)
#     sample, cache_entry = phase6_pick_training_pair(max_audio_sec=MAX_AUDIO_SEC_B1, balanced=True)
#     print(f"  sample: {sample['src_lang']}->{sample['tgt_lang']} | ref='{sample['ref'][:60]}'")

#     # Test use_teacher_text=False path (the broken one)
#     try:
#         with torch.cuda.amp.autocast(dtype=autocast_dtype):
#             loss = text_recovery_step(sample, cache_entry, use_teacher_text=False)
#         print(f"  use_teacher_text=False  loss={float(loss):.4f}  ✓")
#     except Exception as e:
#         print(f"  use_teacher_text=False  ✗ {e}")
#         all_ok = False

#     # Test use_teacher_text=True path (was already working)
#     try:
#         with torch.cuda.amp.autocast(dtype=autocast_dtype):
#             loss = text_recovery_step(sample, cache_entry, use_teacher_text=True)
#         print(f"  use_teacher_text=True   loss={float(loss):.4f}  ✓")
#     except Exception as e:
#         print(f"  use_teacher_text=True   ✗ {e}")
#         all_ok = False

#     print("\n" + "=" * 60)
#     print("VERDICT:", "✓ build_target_labels is correct" if all_ok else "✗ Still broken")
#     return all_ok

# # Build inverse map for round-trip decode check
# _old_to_new_inv = {v: k for k, v in _old_to_new.items()}

# verify_build_target_labels()

In [37]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # CACHE POISONING DIAGNOSTIC — run this to decide if rebuild is needed
# # ═══════════════════════════════════════════════════════════════════════════════

# print("=" * 80)
# print("PHASE 6A CACHE POISONING ANALYSIS")
# print("=" * 80)

# # How many entries to sample
# N_SAMPLES = 10

# # Get random keys from the loaded cache index
# all_keys = list(phase6_cache_index.keys())
# sample_keys = random.sample(all_keys, min(N_SAMPLES, len(all_keys)))

# # Precompute new->old remap tensor for correct decoding
# _new_to_old_tensor = model_student._vocab_remap_to_old

# def correct_decode_string(new_ids):
#     """Remap new IDs -> old IDs, then use processor.batch_decode (correct)."""
#     old_ids = _new_to_old_tensor.to(new_ids.device)[new_ids]
#     return processor.batch_decode(old_ids.unsqueeze(0), skip_special_tokens=True)[0].strip()

# def poisoned_decode_string(new_ids):
#     """Feed new IDs directly to processor (what the buggy cache builder did)."""
#     return processor.batch_decode(new_ids.unsqueeze(0), skip_special_tokens=True)[0].strip()

# bug_count = 0
# ok_count = 0
# ambiguous_count = 0

# for key in sample_keys:
#     entry = phase6_get_cache_entry(key)
#     seq = entry['teacher_text_sequences']
#     stored_str = entry['teacher_text_str']

#     correct_str = correct_decode_string(seq)
#     poisoned_str = poisoned_decode_string(seq)

#     matches_poisoned = (stored_str == poisoned_str)
#     matches_correct = (stored_str == correct_str)

#     status = ""
#     if matches_poisoned and not matches_correct:
#         status = "BUG CONFIRMED"
#         bug_count += 1
#     elif matches_correct and not matches_poisoned:
#         status = "OK"
#         ok_count += 1
#     else:
#         status = "AMBIGUOUS"
#         ambiguous_count += 1

#     print(f"\n[{status}] {key}")
#     print(f"  stored:   '{stored_str[:60]}'")
#     print(f"  correct:  '{correct_str[:60]}'")
#     print(f"  poisoned: '{poisoned_str[:60]}'")
#     print(f"  seq_len={len(seq)}  audio={entry['audio_len_s']:.1f}s  "
#           f"units={len(entry['teacher_unit_sequences'])}")

# print("\n" + "=" * 80)
# print(f"SAMPLES: {bug_count} BUGGED | {ok_count} OK | {ambiguous_count} AMBIGUOUS")
# print("=" * 80)

# if bug_count > 0:
#     print("\n>>> CACHE IS POISONED. teacher_text_str is wrong.")
#     print(">>> BUT teacher_text_sequences and teacher_unit_sequences are CORRECT.")
#     print(">>> FIX: iterate all shards, regenerate teacher_text_str from sequences.")
#     print(">>> NO NEED to rerun teacher model inference.")
# else:
#     print("\n>>> Cache strings are correct. No rebuild needed for text_str.")

In [38]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # IN-PLACE CACHE FIX — regenerate teacher_text_str without teacher model
# # ═══════════════════════════════════════════════════════════════════════════════

# import gc
# import os
# import glob
# from collections import OrderedDict

# _new_to_old_tensor = model_student._vocab_remap_to_old.to('cpu')

# def fix_entry_text_str(entry):
#     """Regenerate teacher_text_str from teacher_text_sequences using correct remap."""
#     seq = entry['teacher_text_sequences']
#     # Remap new IDs -> old IDs, then decode
#     old_ids = _new_to_old_tensor[seq]
#     entry['teacher_text_str'] = processor.batch_decode(
#         old_ids.unsqueeze(0), skip_special_tokens=True
#     )[0].strip()
#     return entry

# # Clear in-memory shard cache to force fresh loads
# phase6_shard_cache = OrderedDict()

# manifest = phase6_load_manifest('train')
# split_name = manifest['split']
# num_shards = manifest['num_shards']
# total_cached = manifest['total_cached']

# print(f"Fixing {total_cached} entries across {num_shards} shards...")

# fixed_total = 0
# for shard_idx in range(num_shards):
#     shard_path = os.path.join(
#         CKPT_DIR,
#         f"{phase6_cache_checkpoint_name(split_name)}_step{shard_idx:06d}.pt",
#     )
#     if not os.path.exists(shard_path):
#         print(f"  WARNING: shard {shard_idx} missing at {shard_path}")
#         continue
    
#     shard_blob = torch.load(shard_path, map_location='cpu', weights_only=False)
#     entries = shard_blob.get('entries', [])
    
#     for entry in entries:
#         entry = fix_entry_text_str(entry)
#         fixed_total += 1
    
#     # Resave fixed shard
#     phase6_save_cache_shard_local(split_name, shard_idx, entries, total_cached)
#     print(f"  Shard {shard_idx:>3}/{num_shards} fixed ({len(entries)} entries)")
    
#     # Aggressive cleanup
#     del shard_blob, entries
#     safe_gc()

# # Update manifest to mark fixed
# manifest['cache_str_fixed'] = True
# phase6_save_manifest_local(split_name, manifest)

# # Push fixed shards to Drive
# if ON_KAGGLE:
#     phase6_rclone_copy_checkpoint_family(
#         [phase6_cache_checkpoint_name(split_name), phase6_cache_manifest_name(split_name)],
#         direction='push',
#     )
#     print("[rclone] Fixed cache pushed to Drive.")

# print(f"\n✅ Done. Fixed {fixed_total} entries. No teacher inference needed.")

In [39]:
# ── Helper: find the latest saved optimizer step for a stage ─────────────────
def phase6_get_resume_step(stage_key):
    """
    Returns the optimizer step stored in the latest checkpoint for this stage.
    Returns 0 if no checkpoint exists (fresh start).
    To extend a run: just increase the STAGE6XX_STEPS constant and re-run the cell.
    The function will pick up from the last saved checkpoint automatically.
    """
    ckpt = phase6_load_latest_local_checkpoint(f'phase6_{stage_key}')
    if ckpt is None:
        print(f'  [{stage_key}] No checkpoint found — starting fresh.')
        return 0
    saved_step   = ckpt.get('optimizer_step', 0)
    saved_total  = ckpt.get('steps_total',    '?')
    print(f'  [{stage_key}] Checkpoint found at optimizer step {saved_step}/{saved_total}')
    return saved_step

In [40]:
# heed

In [41]:
# ── Load 6b2 step 1350 weights into student, then launch 6c fresh ─────────────

# Load the specific step 1350 checkpoint by path (not latest)
ckpt_path = f'{CKPT_DIR}/phase6_6b2_step001350.pt'
ckpt_6b2  = torch.load(ckpt_path, map_location='cpu', weights_only=False)

assert ckpt_6b2['optimizer_step'] == 1350, f"Expected step 1350, got {ckpt_6b2['optimizer_step']}"

model_student.text_decoder.load_state_dict(ckpt_6b2['text_decoder'])
model_student.speech_encoder.load_state_dict(ckpt_6b2['speech_encoder'])
print(f"✓ Loaded 6b2 weights from step {ckpt_6b2['optimizer_step']} "
      f"(Text-ChrF=40.47, ASR-ChrF=37.06)")

del ckpt_6b2
safe_gc()

✓ Loaded 6b2 weights from step 1350 (Text-ChrF=40.47, ASR-ChrF=37.06)


In [42]:
phase6_quick_eval('pre 6c test', max_samples=16)

[MMS-ASR] Loading facebook/mms-1b-all lang=ben...


preprocessor_config.json:   0%|          | 0.00/254 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

[MMS-ASR] ara ready.
[MMS-ASR] Loading facebook/mms-1b-all lang=hin...


Loading weights:   0%|          | 0/1096 [00:00<?, ?it/s]

adapter.hin.safetensors:   0%|          | 0.00/9.29M [00:00<?, ?B/s]

[MMS-ASR] hin ready.
  [pre 6c test] quick Text-ChrF: 40.47 | ASR-ChrF: 37.07


(40.474533771086534, 37.06767432557287)

In [43]:
# !rm -rf checkpoints/phase6_6c_step000050.pt
# !ls checkpoints

In [ ]:
  ensure_teacher_loaded()

  phase6_logs["6c"] = run_phase6c_corrected(
      model_student=model_student,
      model_teacher=model_teacher,
      student_device=student_device,
      teacher_device=teacher_device,
      phase6_pick_training_pair=phase6_pick_training_pair,
      phase6_quick_eval=phase6_quick_eval,
      phase6_cache_index=phase6_cache_index,
      steps=STAGE6C_STEPS,          # 700 or 1100
      grad_accum=GRAD_ACCUM,        # 8
      max_audio_sec=MAX_AUDIO_SEC_C,  # 12
      base_lr=8e-5,
      dur_lr=1e-4,
      log_every=LOG_EVERY,
      eval_every=EVAL_EVERY,
      save_every=SAVE_EVERY,
      ckpt_dir=CKPT_DIR,
      autocast_dtype=autocast_dtype,
      use_kd_supplement=True,   # set False if GPU1 is tight
  )

  phase6_quick_eval("stage6c_done", max_samples=16)

  T2U mode: full native training
  Cast 178 trainable FP16 params to FP32

Mini-test: 50 steps | Student conditioning | LR 8e-5 | Smooth-L1 length
If ASR-ChrF stays ~37 or rises, the doc's approach is viable.

  mini opt  10/50 | loss=25.8312 | soft=0.9740 | hard=8.3228 | len=227.5000
  mini opt  20/50 | loss=7.8293 | soft=1.0128 | hard=8.5721 | len=46.5000
  [mini6c_doc_step25] quick Text-ChrF: 40.47 | ASR-ChrF: 26.25
  mini opt  30/50 | loss=4.4190 | soft=0.7898 | hard=6.9837 | len=18.5000
  mini opt  40/50 | loss=5.6782 | soft=1.1953 | hard=8.7033 | len=23.5000
  mini opt  50/50 | loss=3.0656 | soft=0.7238 | hard=5.9375 | len=8.5000
  [mini6c_doc_step50] quick Text-ChrF: 40.47 | ASR-ChrF: 21.56

Mini-test complete.


In [ ]:
heed

In [ ]:
# # ═══════════════════════════════════════════════════════════════════════════════
# # PHASE 6C MINI TEST — 50 optimizer steps to validate the teacher-conditioning fix
# # ═══════════════════════════════════════════════════════════════════════════════

# ensure_teacher_loaded()
# mark_t2u_trainable_full()  # or selective, your choice
# ensure_trainable_fp32()

# # Use higher LR for quick signal
# optimizer = torch.optim.AdamW(
#     make_t2u_param_groups(model_student, base_lr=2e-4, dur_lr=3e-4,
#                           scalar_lr=3e-4, head_lr=2e-4),
#     betas=(0.9, 0.98),
# )
# scheduler = make_cosine_scheduler(optimizer, 50)
# scaler    = torch.cuda.amp.GradScaler(enabled=torch.cuda.is_available())

# model_student.train()
# optimizer.zero_grad(set_to_none=True)

# print("Mini-test: 50 steps with TEACHER conditioning fed to student T2U")
# print("If ASR-ChrF stays flat or rises instead of crashing, the fix is viable.\n")

# for micro_step in range(50 * GRAD_ACCUM):
#     sample, cache_entry    = phase6_pick_training_pair(max_audio_sec=MAX_AUDIO_SEC_C, balanced=True)
#     teacher_text_sequences = cache_entry["teacher_text_sequences"].unsqueeze(0)
#     audio_inputs_student   = phase6_prepare_audio_inputs(sample, student_device)
#     audio_inputs_teacher   = {k: v.to(teacher_device) for k, v in audio_inputs_student.items()}

#     try:
#         # ── Teacher path (GPU1) ───────────────────────────────────────────
#         with torch.no_grad():
#             teacher_cond = build_t2u_conditioning_from_sequences(
#                 model_teacher,
#                 input_features=audio_inputs_teacher["input_features"],
#                 attention_mask=audio_inputs_teacher.get("attention_mask"),
#                 text_sequences=teacher_text_sequences.to(teacher_device),
#             )
#             with torch.cuda.amp.autocast(dtype=autocast_dtype):
#                 teacher_t2u = model_teacher.t2u_model(
#                     inputs_embeds=teacher_cond["t2u_input_embeds"],
#                     attention_mask=teacher_cond["t2u_attention_mask"],
#                     char_input_ids=teacher_cond["t2u_char_input_ids"],
#                     char_count_per_id=teacher_cond["t2u_char_count_per_id"],
#                     return_dict=True,
#                 )

#         # ── THE FIX: Teacher conditioning → Student T2U ───────────────────
#         teacher_cond_student = {
#             k: v.to(student_device).detach()
#             for k, v in teacher_cond.items()
#         }

#         with torch.cuda.amp.autocast(dtype=autocast_dtype):
#             student_t2u = model_student.t2u_model(
#                 inputs_embeds=teacher_cond_student["t2u_input_embeds"],
#                 attention_mask=teacher_cond_student["t2u_attention_mask"],
#                 char_input_ids=teacher_cond_student["t2u_char_input_ids"],
#                 char_count_per_id=teacher_cond_student["t2u_char_count_per_id"],
#                 return_dict=True,
#             )

#             teacher_t2u.last_hidden_state = (
#                 teacher_t2u.last_hidden_state.to(student_device)
#                 if hasattr(teacher_t2u, "last_hidden_state") and teacher_t2u.last_hidden_state is not None
#                 else None
#             )
#             if hasattr(teacher_t2u, "logits") and teacher_t2u.logits is not None:
#                 teacher_t2u.logits = teacher_t2u.logits.to(student_device)
#             teacher_t2u.padding_mask = teacher_t2u.padding_mask.to(student_device)

#             t2u_soft, t2u_hard, t2u_len = t2u_overlap_losses(student_t2u, teacher_t2u)
#             loss = 0.60 * t2u_soft + 0.30 * t2u_hard + 0.10 * t2u_len

#         scaler.scale(loss / GRAD_ACCUM).backward()

#     except RuntimeError as e:
#         if "out of memory" in str(e).lower():
#             safe_gc()
#             raise RuntimeError("OOM in mini-test — try selective mode")
#         raise

#     if (micro_step + 1) % GRAD_ACCUM == 0:
#         opt_step = (micro_step + 1) // GRAD_ACCUM
#         scaler.unscale_(optimizer)
#         torch.nn.utils.clip_grad_norm_(
#             [p for p in model_student.parameters() if p.requires_grad],
#             MAX_GRAD_NORM,
#         )
#         scaler.step(optimizer)
#         scaler.update()
#         optimizer.zero_grad(set_to_none=True)
#         scheduler.step()

#         if opt_step % 10 == 0:
#             print(f"  mini opt {opt_step:>3}/50 | loss={loss.item():.4f} | soft={t2u_soft.item():.4f} | hard={t2u_hard.item():.4f}")

#         if opt_step == 25 or opt_step == 50:
#             phase6_quick_eval(f'mini6c_step{opt_step}', max_samples=16)

# print("\nMini-test complete.")

In [ ]:
# STAGE6C_STEPS    = 1100    # — T2U distillation

# # ── Stage 6C ──────────────────────────────────────────────────────────────────
# phase6_logs['6c'] = run_t2u_recovery_stage(
#     stage_key        = '6c',
#     title            = 'Native T2U recovery with teacher KD',
#     steps            = STAGE6C_STEPS,
#     max_audio_sec    = MAX_AUDIO_SEC_C,
#     resume_from_step = phase6_get_resume_step('6c'),
# )
# phase6_quick_eval('stage6c_done', max_samples=16)

In [ ]:
STAGE6D_STEPS    = 600
# STAGE6D_ENABLED  = True
# ── Stage 6D ──────────────────────────────────────────────────────────────────
if STAGE6D_ENABLED:
    phase6_logs['6d'] = run_joint_polish_stage(
        stage_key        = '6d',
        title            = 'Joint polish: text LoRA + native T2U',
        steps            = STAGE6D_STEPS,
        max_audio_sec    = MAX_AUDIO_SEC_D,
        resume_from_step = phase6_get_resume_step('6d'),
    )
    phase6_quick_eval('stage6d_done', max_samples=16)
else:
    phase6_logs['6d'] = []
    print('Stage 6D skipped by configuration.')

In [ ]:
# phase6_logs['6b1'] = run_text_recovery_stage(
#     stage_key='6b1',
#     title='Text decoder warmup (LoRA only)',
#     steps=STAGE6B1_STEPS,
#     max_audio_sec=MAX_AUDIO_SEC_B1,
#     text_lr=5e-4,
#     speech_lr=None,
#     kd_prob=PHASE6_TEXT_KD_PROB,
# )

# phase6_quick_eval('stage6b1_done', max_samples=8)


In [ ]:
# phase6_logs['6b2'] = run_text_recovery_stage(
#     stage_key='6b2',
#     title='Speech encoder + text decoder recovery (LoRA)',
#     steps=STAGE6B2_STEPS,
#     max_audio_sec=MAX_AUDIO_SEC_B2,
#     text_lr=8e-5,
#     speech_lr=4e-5,
#     kd_prob=PHASE6_TEXT_KD_PROB,
# )

# phase6_quick_eval('stage6b2_done', max_samples=8)


In [ ]:
# phase6_logs['6c'] = run_t2u_recovery_stage(
#     stage_key='6c',
#     title='Native T2U recovery with teacher KD',
#     steps=STAGE6C_STEPS,
#     max_audio_sec=MAX_AUDIO_SEC_C,
# )

# phase6_quick_eval('stage6c_done', max_samples=8)


In [ ]:
# if STAGE6D_ENABLED:
#     phase6_logs['6d'] = run_joint_polish_stage(
#         stage_key='6d',
#         title='Joint polish: text LoRA + native T2U',
#         steps=STAGE6D_STEPS,
#         max_audio_sec=MAX_AUDIO_SEC_D,
#     )
#     phase6_quick_eval('stage6d_done', max_samples=8)
# else:
#     phase6_logs['6d'] = []
#     print('Stage 6D skipped by configuration.')


In [ ]:
print('\n' + '=' * 80)
print('Merging LoRA adapters and saving Phase 6 model')
print('=' * 80)

if model_teacher is not None:
    del model_teacher
    model_teacher = None
    safe_gc()

if hasattr(model_student.speech_encoder, 'merge_and_unload'):
    model_student.speech_encoder = model_student.speech_encoder.merge_and_unload()
if hasattr(model_student.text_decoder, 'merge_and_unload'):
    model_student.text_decoder = model_student.text_decoder.merge_and_unload()

model_student.eval()
sync_model_config(model_student)
save_model_to_drive(model_student, processor, PHASE6_MODEL_NAME)

model_p6 = model_student
print(f'Phase 6 model saved as: {PHASE6_MODEL_NAME}')
gpu_mem()


In [ ]:
def phase6_ema(values, alpha=0.05):
    if not values:
        return []
    out = [values[0]]
    for v in values[1:]:
        out.append(alpha * v + (1 - alpha) * out[-1])
    return out


fig, axes = plt.subplots(2, 2, figsize=(16, 10))
fig.suptitle('Phase 6 Recovery Training', fontsize=14, fontweight='bold')

ax = axes[0, 0]
if phase6_logs['6b1']:
    vals = [row['loss'] for row in phase6_logs['6b1']]
    ax.plot(vals, alpha=0.20, color='#1976D2', lw=0.6, label='raw')
    ax.plot(phase6_ema(vals), color='#1976D2', lw=2.0, label='ema')
ax.set_title('Stage 6B1: Text Decoder Warmup', fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.grid(alpha=0.3)
ax.legend()

ax = axes[0, 1]
if phase6_logs['6b2']:
    vals = [row['loss'] for row in phase6_logs['6b2']]
    ax.plot(vals, alpha=0.20, color='#388E3C', lw=0.6, label='raw')
    ax.plot(phase6_ema(vals), color='#388E3C', lw=2.0, label='ema')
ax.set_title('Stage 6B2: Speech + Text LoRA', fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.grid(alpha=0.3)
ax.legend()

ax = axes[1, 0]
if phase6_logs['6c']:
    soft = [row['t2u_soft'] for row in phase6_logs['6c']]
    hard = [row['t2u_hard'] for row in phase6_logs['6c']]
    ax.plot(phase6_ema(soft), color='#8E24AA', lw=2.0, label='soft KD')
    ax.plot(phase6_ema(hard), color='#FB8C00', lw=2.0, label='hard KD')
ax.set_title('Stage 6C: Native T2U KD', fontweight='bold')
ax.set_xlabel('Step')
ax.set_ylabel('Loss')
ax.grid(alpha=0.3)
ax.legend()

ax = axes[1, 1]
if phase6_eval_history:
    xs = list(range(1, len(phase6_eval_history) + 1))
    ys = [row['chrf'] for row in phase6_eval_history]
    labels = [row['tag'] for row in phase6_eval_history]
    ax.plot(xs, ys, marker='o', color='#D81B60', lw=2.0)
    ax.set_xticks(xs)
    ax.set_xticklabels(labels, rotation=45, ha='right')
ax.set_title('Quick ASR-ChrF Checks', fontweight='bold')
ax.set_xlabel('Checkpoint')
ax.set_ylabel('ASR-ChrF')
ax.grid(alpha=0.3)

plt.tight_layout()
save_figure(fig, 'phase6_lora_t2u_training.png')
plt.show()


In [ ]:
p6_bench = load_latest_checkpoint(PHASE6_BENCHMARK_NAME)
if p6_bench:
    p6_results = p6_bench['results']
    p6_summary = p6_bench['summary']
    p6_detailed = p6_bench.get('detailed_summary')
    if not p6_detailed:
        p6_detailed = compute_detailed_summary(p6_results, 'P6_LoRA_T2U', p6_summary['params_M'])
else:
    p6_results, p6_summary = run_benchmark_asr(
        model_p6,
        eval_samples,
        'P6_LoRA_T2U',
        save_n=4,
    )
    p6_detailed = compute_detailed_summary(p6_results, 'P6_LoRA_T2U', p6_summary['params_M'])
    save_checkpoint(
        {
            'results': p6_results,
            'summary': p6_summary,
            'detailed_summary': p6_detailed,
        },
        PHASE6_BENCHMARK_NAME,
        0,
    )

store_summary(p6_summary)
store_detailed_summary(p6_detailed)
print_detailed_summary_table('P6_LoRA_T2U')
plot_phase_comparison()
plot_detailed_phase_comparison()


In [ ]:
safe_gc()
print_model_breakdown(model_p6, 'Phase 6 recovered model')
gpu_mem()
print('Phase 6 is ready for Phase 7.')


In [ ]:
# Phase 6 complete.


---
## Phase 7: Textless Inference + Full Comprehensive Benchmark
Evaluation of the final ~673M textless model:
1. Translation quality — ASR-ChrF, all 5 languages, bidirectional
2. Voice cloning — ECAPA cosine similarity (input vs output speaker)
3. Long-form audio — 5s, 15s, 30s, 60s (chunked inference for >25s)
4. Audio quality — UTMOS naturalness score
5. Speed — RTF comparison vs V1 and teacher


In [ ]:
# ── Load final model ──────────────────────────────────────────────────────────
model_final, processor = load_model_from_drive(PHASE6_MODEL_NAME, device_map=None)

# Rebuild CIF + speaker adapter on model_final if needed
if not hasattr(model_final,'cif_connector') or model_final.cif_connector is None:
    model_final.cif_connector  = CIFConnector(d_model=hidden, n_langs=n_langs+5)
    model_final.speaker_adapter = SpeakerAdapter()

p6b_final = load_latest_checkpoint('phase6b_e2e')
if p6b_final:
    model_final.cif_connector.load_state_dict(p6b_final.get('cif_connector', {}), strict=False)
    model_final.speaker_adapter.load_state_dict(p6b_final.get('speaker_adapter', {}), strict=False)
    print('Final CIF + speaker adapter weights loaded.')

model_final.eval()
model_final = _consolidate_to_single_gpu(model_final)
device_final = torch.device('cuda:0')
print_model_breakdown(model_final, 'FINAL ~673M Textless Model')
gpu_mem()


In [ ]:

def translate_longform(mdl, audio_wav, tgt_lang, chunk_s=25, overlap_s=2, sr=16000):
    chunk_len   = chunk_s * sr
    overlap_len = overlap_s * sr
    hop_len     = chunk_len - overlap_len
    chunks, pos = [], 0
    while pos < len(audio_wav):
        chunk = audio_wav[pos : pos + chunk_len]
        if len(chunk) < sr // 2:
            break
        chunks.append(chunk)
        pos += hop_len
    print(f'Long-form {len(audio_wav)/sr:.1f}s → {len(chunks)} chunk(s) × {chunk_s}s')
    outputs = []
    for i, chunk in enumerate(chunks):
        wav_out, rtf, _ = run_s2st(mdl, chunk, tgt_lang)
        if i > 0 and len(wav_out) > overlap_len // 2:
            wav_out = wav_out[overlap_len // 2:]
        outputs.append(wav_out)
        print(f'  Chunk {i+1}/{len(chunks)} RTF={rtf:.3f}')
    return np.concatenate(outputs) if outputs else np.zeros(sr)

print('✓ Textless inference ready.')

DEBUG T2U

In [ ]:
# ── DEMO: Listen to voice-cloned translation ──────────────────────────────────
demo = eval_samples[0]
print(f'Source (EN): {demo["src_text"]}')
play(demo['wav'], 16000, 'Input (English)')

tgt = 'ben'
print(f'\nTranslating EN→{tgt.upper()}...')
try:
    wav_out, rtf, _ = run_s2st(model_final, demo['wav'], tgt_lang=tgt)
    
    hyp = asr_transcribe(wav_out, tgt)
    print(f'  ASR: {hyp[:120]}')
    print(f'  RTF: {rtf:.3f}')
    play(wav_out, 16000, f'Output ({tgt}, voice-cloned)')
    save_audio(wav_out, 16000, f'demo_{tgt}.wav')
except Exception as e:
    print(f'  Error: {e}')


In [ ]:
# ── BENCHMARK 1: Translation quality — all 5 languages, bidirectional ─────────
p7_trans_ckpt = load_latest_checkpoint('phase7_translation')
if p7_trans_ckpt:
    trans_results = p7_trans_ckpt['results']
    print('Loaded translation results.')
else:
    trans_results = {}
    model_final.eval()
    
    # Group eval_samples by language pair
    from collections import defaultdict
    samples_by_pair = defaultdict(list)
    for s in eval_samples:
        pair_key = f"{s['src_lang']}→{s['tgt_lang']}"
        samples_by_pair[pair_key].append(s)
    
    for pair_key, pair_samples in samples_by_pair.items():
        print(f'\nBenchmarking {pair_key} ({len(pair_samples)} samples)...')
        pair_res = []
        for s in pair_samples:
            try:
                wav_out, rtf, _ = run_s2st(model_final, s['wav'], tgt_lang=s['tgt_lang'])
                hyp  = asr_transcribe(wav_out, s['tgt_lang'])
                chrf = compute_chrf(hyp, s['ref'])
                bleu = compute_bleu(hyp, s['ref'])
                pair_res.append(dict(id=s['id'],hyp=hyp,ref=s['ref'],chrf=chrf,bleu=bleu,rtf=rtf))
            except Exception as e:
                print(f'  Error: {e}')
                pair_res.append(dict(id=s.get('id','?'),hyp='',ref=s.get('ref',''),chrf=0,bleu=0,rtf=0))
        
        trans_results[pair_key] = dict(
            results=pair_res,
            avg_chrf=float(np.mean([r['chrf'] for r in pair_res])),
            avg_bleu=float(np.mean([r['bleu'] for r in pair_res])),
            avg_rtf =float(np.mean([r['rtf']  for r in pair_res])),
        )
        print(f'  {pair_key}: ASR-ChrF={trans_results[pair_key]["avg_chrf"]:.2f} '
              f'ASR-BLEU={trans_results[pair_key]["avg_bleu"]:.2f} RTF={trans_results[pair_key]["avg_rtf"]:.4f}')
    
    save_checkpoint({'results': trans_results}, 'phase7_translation', 0)

print('\n--- Translation Quality (ASR-ChrF/BLEU) ---')
print(f'  {"Pair":<15} {"ASR-ChrF":>10} {"ASR-BLEU":>10} {"RTF":>7}')
for pair, res in trans_results.items():
    print(f'  {pair:<15} {res["avg_chrf"]:>10.2f} {res["avg_bleu"]:>10.2f} {res["avg_rtf"]:>7.4f}')


In [ ]:
# ── BENCHMARK 2: Voice cloning — ECAPA speaker similarity ────────────────────
p7_spk_ckpt = load_latest_checkpoint('phase7_speaker_sim')
if p7_spk_ckpt:
    spk_results = p7_spk_ckpt['results']; print('Loaded speaker sim results.')
else:
    spk_results = []
    # Test on subset of language pairs
    test_pairs = [('eng','ben'),('eng','hin'),('eng','cmn'),('eng','arb')]
    for src_lang, tgt_lang in test_pairs:
        pair_samples = [s for s in eval_samples if s['src_lang']==src_lang and s['tgt_lang']==tgt_lang][:10]
        print(f'  Speaker sim {src_lang}→{tgt_lang}...')
        for s in pair_samples:
            try:
                wav_out, rtf, _ = run_s2st(model_final, s['wav'], tgt_lang=tgt_lang)
                src_emb = extract_speaker_emb(s['wav'])
                out_emb = extract_speaker_emb(wav_out) if len(wav_out)>800 else src_emb*0
                sim = F.cosine_similarity(src_emb.unsqueeze(0), out_emb.unsqueeze(0)).item()
                spk_results.append({'id':s['id'],'pair':f'{src_lang}→{tgt_lang}',
                                    'speaker_sim':sim,'rtf':rtf})
                print(f'    {s["id"]}: sim={sim:.3f}')
            except Exception as e:
                print(f'    Error: {e}')
    save_checkpoint({'results': spk_results}, 'phase7_speaker_sim', 0)

if spk_results:
    avg_sim = np.mean([r['speaker_sim'] for r in spk_results])
    qual = ('Excellent' if avg_sim>0.85 else 'Good' if avg_sim>0.70
            else 'Acceptable' if avg_sim>0.55 else 'Poor')
    print(f'\nVoice cloning — avg ECAPA sim: {avg_sim:.3f}  [{qual}]')
    print(f'  Target: 0.65–0.78  |  SeamlessExpressive: ~0.80')


In [ ]:
# ── BENCHMARK 3: Long-form audio (PLAN.md Section 2.3) ───────────────────────
p7_lf_ckpt = load_latest_checkpoint('phase7_longform')
if p7_lf_ckpt:
    longform_results = p7_lf_ckpt['results']; print('Loaded long-form results.')
else:
    longform_results = {}
    DURATIONS = [5, 15, 30, 60]
    base_wavs = [s['wav'] for s in eval_samples[:8]]
    base_refs = [s['ref'] for s in eval_samples[:8]]

    def make_test_audio(target_s, wavs, sr=16000):
        combined = np.concatenate(wavs)
        tlen = target_s * sr
        if len(combined) < tlen:
            reps = math.ceil(tlen/len(combined))
            combined = np.tile(combined, reps)
        return combined[:tlen]

    model_final.eval()
    for dur_s in DURATIONS:
        print(f'\nLong-form {dur_s}s...')
        test_wav = make_test_audio(dur_s, base_wavs)
        test_ref = ' '.join(base_refs)
        chrfs, rtfs = [], []
        for trial in range(3):
            try:
                if dur_s <= 25:
                    wav_out, rtf, _ = run_s2st(model_final, test_wav, tgt_lang='ben')
                else:
                    t0 = time.time()
                    wav_out = translate_longform(model_final, test_wav, tgt_lang='ben')
                    rtf = (time.time()-t0)/dur_s
                if len(wav_out)>800:
                    hyp = asr_transcribe(wav_out, 'ben')
                    chrfs.append(compute_chrf(hyp, test_ref[:300]))
                    rtfs.append(rtf)
            except Exception as e:
                print(f'  Trial {trial+1} error: {e}')
        longform_results[dur_s] = {
            'duration_s': dur_s, 'method': 'direct' if dur_s<=25 else 'chunked_25s+2s_overlap',
            'avg_chrf': float(np.mean(chrfs)) if chrfs else 0,
            'avg_rtf':  float(np.mean(rtfs))  if rtfs  else 0,
        }
        print(f'  {dur_s}s: ChrF={longform_results[dur_s]["avg_chrf"]:.2f} RTF={longform_results[dur_s]["avg_rtf"]:.3f}')
    save_checkpoint({'results': longform_results}, 'phase7_longform', 0)

print('\nLong-form results:')
for dur, res in sorted(longform_results.items()):
    print(f'  {dur}s [{res["method"]}]: ChrF={res["avg_chrf"]:.2f}  RTF={res["avg_rtf"]:.3f}')


In [ ]:
# ── FINAL COMPREHENSIVE VISUALISATION (paper figures) ─────────────────────────
fig = plt.figure(figsize=(20, 16))
fig.suptitle('Textless SeamlessM4T v2 (~673M): Comprehensive Benchmark - 5 Languages',
             fontsize=14, fontweight='bold', y=0.99)

# 1: Parameter evolution
ax1 = fig.add_subplot(3,3,1)
phase_names  = ['Teacher\n1805M','V1\n1039M','Vocab5L\n824M','Enc16L\n630M',
                'LaCoT2U\n542M','Textless\n673M']
phase_params = [1805, 1039, 824, 630, 542, 673]
colors_pb = ['#9E9E9E']*5 + ['#4CAF50']
bars = ax1.bar(range(len(phase_names)), phase_params, color=colors_pb, alpha=0.85, edgecolor='white')
bars[-1].set_edgecolor('#2E7D32'); bars[-1].set_linewidth(2)
ax1.set_xticks(range(len(phase_names))); ax1.set_xticklabels(phase_names, fontsize=7)
ax1.set_ylabel('Parameters (M)'); ax1.set_title('Model Size Evolution', fontweight='bold')
for bar, v in zip(bars, phase_params):
    ax1.text(bar.get_x()+bar.get_width()/2, bar.get_height()+10, f'{v}M',
             ha='center', va='bottom', fontsize=7, fontweight='bold')

# 2: ASR-ChrF by lang pair (all 8 pairs)
ax2 = fig.add_subplot(3,3,2)
if trans_results:
    pairs_  = list(trans_results.keys())
    chrfs_  = [trans_results[p]['avg_chrf'] for p in pairs_]
    bleus_  = [trans_results[p]['avg_bleu'] for p in pairs_]
    x_  = np.arange(len(pairs_)); w_ = 0.35
    ax2.bar(x_-w_/2, chrfs_, w_, label='ASR-ChrF', color='#2196F3', alpha=0.85)
    ax2.bar(x_+w_/2, bleus_, w_, label='ASR-BLEU', color='#FF9800', alpha=0.85)
    ax2.set_xticks(x_); ax2.set_xticklabels(pairs_, rotation=45, ha='right', fontsize=7)
    ax2.set_title('Translation Quality by Language Pair (ASR)', fontweight='bold')
    ax2.legend(fontsize=8)
    ax2.axhline(35, color='green', ls=':', lw=1.5, alpha=0.7, label='Target')

# 3: Speaker similarity
ax3 = fig.add_subplot(3,3,3)
if spk_results:
    sims_spk = [r['speaker_sim'] for r in spk_results]
    ax3.hist(sims_spk, bins=12, color='#E91E63', alpha=0.8, edgecolor='white')
    for thresh, lbl, col in [(0.85,'Excellent','green'),(0.70,'Good','orange'),(0.55,'Acceptable','red')]:
        ax3.axvline(thresh, color=col, ls='--', lw=1.5, label=f'{lbl}>{thresh}')
    ax3.axvline(np.mean(sims_spk), color='black', ls='-', lw=2,
                label=f'Mean={np.mean(sims_spk):.3f}')
    ax3.set_xlabel('ECAPA Cosine Similarity'); ax3.set_title('Speaker Similarity (Voice Cloning)', fontweight='bold')
    ax3.legend(fontsize=7)

# 4: Long-form quality
ax4 = fig.add_subplot(3,3,4)
if longform_results:
    durs_ = sorted(longform_results.keys())
    lf_ch = [longform_results[d]['avg_chrf'] for d in durs_]
    lf_rt = [longform_results[d]['avg_rtf']  for d in durs_]
    ax4_t = ax4.twinx()
    ax4.plot(durs_, lf_ch, 'o-', color='#4CAF50', lw=2, ms=8, label='ASR-ChrF')
    ax4_t.plot(durs_, lf_rt, 's--', color='#FF5722', lw=2, ms=8, label='RTF')
    ax4.axvline(25, color='gray', ls=':', lw=1.5, label='Chunking boundary')
    ax4.set_xlabel('Duration (s)'); ax4.set_ylabel('ASR-ChrF', color='#4CAF50')
    ax4_t.set_ylabel('RTF', color='#FF5722')
    ax4.set_title('Long-Form: Quality vs Duration', fontweight='bold')
    ax4.legend(loc='upper left', fontsize=8); ax4_t.legend(loc='upper right', fontsize=8)

# 5: RTF comparison
ax5 = fig.add_subplot(3,3,5)
final_rtf = np.mean([v['avg_rtf'] for v in trans_results.values()]) if trans_results else 0.09
spd_labels = ['Teacher\n1805M','V1\n1039M','Textless\n673M']
spd_rtfs   = [0.268, 0.113, final_rtf]
ax5.bar(spd_labels, spd_rtfs, color=['#F44336','#FF9800','#4CAF50'], alpha=0.85, edgecolor='white')
ax5.set_ylabel('RTF (lower=faster)'); ax5.set_title('Inference Speed (RTF)', fontweight='bold')
for i,(l,v) in enumerate(zip(spd_labels,spd_rtfs)):
    ax5.text(i, v+0.003, f'{v:.3f}', ha='center', fontsize=9, fontweight='bold')

# 6: Speaker sim by pair
ax6 = fig.add_subplot(3,3,6)
if spk_results:
    from collections import defaultdict
    pair_sims_ = defaultdict(list)
    for r in spk_results: pair_sims_[r['pair']].append(r['speaker_sim'])
    pn = list(pair_sims_.keys())
    pm = [np.mean(pair_sims_[p]) for p in pn]
    ps = [np.std(pair_sims_[p]) for p in pn]
    ax6.bar(pn, pm, yerr=ps, capsize=5, color='#9C27B0', alpha=0.8, edgecolor='white')
    ax6.axhline(0.65, color='green', ls='--', lw=1.5, label='Target 0.65')
    ax6.set_ylim(0,1); ax6.set_ylabel('Speaker Similarity')
    ax6.set_xticklabels(pn, rotation=30, ha='right', fontsize=7)
    ax6.set_title('Speaker Sim by Language Pair', fontweight='bold'); ax6.legend(fontsize=8)

# 7: Enc pruning ChrF curve (Phase 2)
ax7 = fig.add_subplot(3,3,7)
if 'p2_log' in dir() and p2_log:
    iters7 = [e['iter'] for e in p2_log]; chrfs7 = [e['chrf'] for e in p2_log]
    ax7.plot(iters7, chrfs7, 'o-', color='#FF9800', lw=2, ms=7)
    for e in p2_log:
        ax7.annotate(f'L{e["removed"]}', (e['iter'],e['chrf']),
                     fontsize=6, ha='center', va='bottom')
    ax7.set_xlabel('Pruning iter'); ax7.set_ylabel('ASR-ChrF')
    ax7.set_title('Enc Pruning: ASR-ChrF per Removal', fontweight='bold')
else:
    ax7.text(0.5,0.5,'P2 log not in session', ha='center', va='center', transform=ax7.transAxes)

# 8: Per-language-pair scatter
ax8 = fig.add_subplot(3,3,8)
if trans_results:
    all_chrfs = []
    all_bleus = []
    for pair_key, pair_data in trans_results.items():
        for r in pair_data['results']:
            all_chrfs.append(r['chrf'])
            all_bleus.append(r['bleu'])
    if all_chrfs:
        ax8.scatter(all_bleus, all_chrfs, color='#2196F3', alpha=0.5, s=30, edgecolors='white')
        mu_c = np.mean(all_chrfs)
        ax8.axhline(mu_c, color='red', ls='--', lw=1.5, label=f'Mean ChrF={mu_c:.1f}')
        ax8.set_xlabel('ASR-BLEU'); ax8.set_ylabel('ASR-ChrF')
        ax8.set_title('All Pairs: BLEU vs ChrF per sample', fontweight='bold'); ax8.legend(fontsize=8)

# 9: Architecture comparison table
ax9 = fig.add_subplot(3,3,9)
ax9.axis('off')
tbl_data = [
    ['Component','Original','Textless 673M'],
    ['Text Decoder','867M 24L','0M (removed)'],
    ['lm_head+vocab','~262M','0M (removed)'],
    ['Speech Encoder','635M 24L','~441M 16L'],
    ['T2U Model','262M 6+6L','~175M 4+4L'],
    ['CIF Connector','—','~5M (NEW)'],
    ['Speaker Adapter','—','~0.1M (NEW)'],
    ['Vocoder','41.9M','41.9M (frozen)'],
    ['TOTAL','1805M','~673M'],
]
tbl = ax9.table(cellText=tbl_data[1:], colLabels=tbl_data[0],
                cellLoc='center', loc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(8); tbl.scale(1.2, 1.5)
for j in range(3):
    tbl[len(tbl_data)-1, j].set_facecolor('#C8E6C9')
    tbl[len(tbl_data)-1, j].set_text_props(fontweight='bold')
tbl[1,2].set_facecolor('#FFCDD2'); tbl[2,2].set_facecolor('#FFCDD2')
ax9.set_title('Architecture Comparison', fontweight='bold', pad=10)

plt.tight_layout(rect=[0,0,1,0.98])
save_figure(fig, 'phase7_comprehensive_benchmark.png')
plt.show()
print('✓ Comprehensive benchmark figure saved (5 languages, ASR metrics).')


In [ ]:
# ── FINAL PAPER TABLE ─────────────────────────────────────────────────────────
print('\n' + '='*80)
print('  FINAL RESULTS — Textless SeamlessM4T v2 ~673M')
print('  Target: INTERSPEECH 2026 · IWSLT 2026 Cross-Lingual Voice Cloning Track')
print('='*80)

avg_chrf_final = np.mean([v['avg_chrf'] for v in trans_results.values()]) if trans_results else 0
avg_bleu_final = np.mean([v['avg_bleu'] for v in trans_results.values()]) if trans_results else 0

print('\n[Table 1: Parameter Reduction]')
print(f'  Teacher (1805M) → V1 (1039M) → Textless (673M)')
print(f'  Compression from teacher: {(1-673/1805)*100:.1f}%')
print(f'  Compression from V1:      {(1-673/1039)*100:.1f}%')

print('\n[Table 2: Translation Quality - All Language Pairs]')
print(f'  {"Pair":<15} {"ASR-ChrF":>10} {"ASR-BLEU":>10} {"RTF":>8}')
for pair, res in sorted(trans_results.items()):
    print(f'  {pair:<15} {res["avg_chrf"]:>10.2f} {res["avg_bleu"]:>10.2f} {res["avg_rtf"]:>8.4f}')
print(f'  {"Average":<15} {avg_chrf_final:>10.2f} {avg_bleu_final:>10.2f}')

print('\n[Table 3: Voice Cloning]')
if spk_results:
    avg_sim = np.mean([r['speaker_sim'] for r in spk_results])
    qual = 'Excellent' if avg_sim>0.85 else 'Good' if avg_sim>0.70 else 'Acceptable' if avg_sim>0.55 else 'Poor'
    print(f'  ECAPA Speaker Similarity: {avg_sim:.3f}  [{qual}]')
    print(f'  Target: 0.65–0.78  (SeamlessExpressive: ~0.80)')

print('\n[Table 4: Speed]')
final_rtf = np.mean([v['avg_rtf'] for v in trans_results.values()]) if trans_results else 0.09
print(f'  Teacher RTF: 0.268 | V1 RTF: 0.113 | Textless RTF: {final_rtf:.3f}')
if final_rtf > 0:
    print(f'  Speedup vs teacher: {0.268/final_rtf:.1f}×')

print('\n[Table 5: Long-Form Support]')
for dur, res in sorted(longform_results.items()):
    print(f'  {dur}s [{res["method"]}]: ASR-ChrF={res["avg_chrf"]:.2f}  RTF={res["avg_rtf"]:.3f}')

print('\n' + '='*80)

# Store final summary
final_summary = dict(
    label='P_Final_Textless_673M',
    params_M=673.0,
    avg_bleu=avg_bleu_final,
    avg_chrf=avg_chrf_final,
    avg_rtf=final_rtf,
    speaker_sim=np.mean([r['speaker_sim'] for r in spk_results]) if spk_results else 0,
    n=sum(len(v['results']) for v in trans_results.values()),
)
store_summary(final_summary)
plot_phase_comparison()
plot_size_vs_quality()

In [ ]:
# Upload all artefacts
if ON_KAGGLE:
    subprocess.run(f'rclone copy "{AUDIO_DIR}/" "{GDRIVE_ROOT}/audio/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M', shell=True)
    subprocess.run(f'rclone copy "{FIG_DIR}/" "{GDRIVE_ROOT}/figures/" --transfers=8 --multi-thread-streams=4 --drive-chunk-size=64M', shell=True)
    print('[rclone] Audio + figures synced to Drive.')

session_status()
print('\n✓ Phase 7 complete. All results persisted to Drive.')

In [ ]:
!rclone copy /kaggle/working/ gdrive:seamTL